# ⚛️ DFT-Flow — Quantum-Chemical Characterisation of Drug-like Molecules
### BinFoLab Computational Pipeline Series · v1.0

**DFT-Flow** takes molecules as SMILES or 3D structures and returns the density-functional-theory (DFT) results that medicinal-chemistry and natural-product papers report: optimised geometries, vibrational frequencies and IR spectra, thermochemistry, frontier molecular orbitals (FMOs), conceptual-DFT reactivity descriptors, Hirshfeld charges and condensed Fukui functions, molecular electrostatic potential (MEP) surfaces with quantitative surface descriptors, and TD-DFT UV-Vis spectra. Every figure is saved at 600 DPI, every table as CSV/XLSX, and the run closes with an auto-written methods paragraph, a PDF report and a ZIP archive.

It is built on **PySCF** with **GPU4PySCF** acceleration, so on a Colab GPU it handles typical drug-sized molecules (roughly 20–60 atoms) at hybrid-DFT level without commercial software.

---

#### Workflow

| Stage | Part | Method | Main outputs |
|---|---|---|---|
| Input & checks | 3 | RDKit sanitisation, charge/multiplicity parity, basis-set coverage, cost estimate | validated molecule table, 2D grid |
| Engine & benchmark | 4–5 | GPU/CPU backend test; reference molecules vs experiment | backend report, validation table |
| Conformer search | 6 | ETKDGv3 → MMFF94 → GFN2-xTB (ALPB) | lowest-energy conformer, Boltzmann populations |
| Geometry optimisation | 7 | DFT-D + geomeTRIC (TRIC coordinates), resumable | optimised XYZ, convergence traces |
| Frequencies | 8 | Analytic Hessian, RRHO thermochemistry, analytic IR intensities | imaginary-mode check, IR spectra, ZPE/H/G |
| Electronic structure | 9 | Higher-basis single point, N±1 vertical states | HOMO/LUMO, gap, χ, η, S, ω, ω⁺, ω⁻ |
| Local reactivity | 10 | Hirshfeld & Mulliken charges, condensed Fukui f⁺/f⁻/f⁰, dual descriptor | reactive-site maps |
| Surfaces | 11 | Density/orbital/ESP cubes, 0.001 au isosurface | FMO and MEP renders, Murray–Politzer descriptors, 3D viewer |
| Excited states | 12 | TD-DFT (RPA or TDA) | UV-Vis spectrum, λmax, transition assignments |
| Comparison | 13 | Cross-molecule statistics | master table, ladder/heatmap/scatter |
| Export | 14 | SDF/XYZ/Molden/cube, methods text, PDF, ZIP | publication package |

#### How to use
1. **Runtime ▸ Change runtime type ▸ GPU** (T4 works; L4/A100 are faster). CPU also works for small molecules but is much slower.
2. Run Part 1, then edit only **Part 2 (settings)** and **Part 3 (molecules)**.
3. Run the rest top to bottom (**Runtime ▸ Run after**). Each stage checkpoints per molecule, so if Colab disconnects, re-run from the top: finished work is skipped and an interrupted optimisation resumes from its last geometry. Turn on *Save to Google Drive* in Part 2 for runs that may outlive a session.

#### Scope and honest limits
- Designed for **closed- and open-shell organic / drug-like molecules** (H–Rn with def2 basis sets). Not for periodic solids, surfaces or nanomaterial slabs.
- Results are single-conformer, harmonic, implicit-solvent quantities. They support interpretation and ranking; they do not replace experiment.

---
## Part 1 — Environment Setup

### 📘 For Your Read — Cell 1.1: Runtime check and installation

**What it does.** Detects whether a GPU is attached, reads the CUDA driver version, and installs the scientific stack: PySCF (DFT engine), geomeTRIC (optimiser), pyscf-dispersion (D3/D4 corrections), tblite (GFN2-xTB), ASE, RDKit, py3Dmol, scikit-image and reportlab. On a GPU runtime it also installs the matching **GPU4PySCF** build (CUDA 12 or 13).

**What you should see.** The GPU name and memory (e.g. *Tesla T4, 15 GB*) and a version table. Installation takes about 1–3 minutes.

**Red flags.** *"No GPU detected"* means you are on a CPU runtime; switch runtime type before continuing unless your molecules are small. If GPU4PySCF fails to install, the notebook falls back to CPU automatically and says so.

In [ ]:
#@title Cell 1.1 — Runtime check & installation
import subprocess, sys, re, time
_t0 = time.time()
def _sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

_q = _sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
HAS_NVIDIA = (_q.returncode == 0) and bool(_q.stdout.strip())
CUDA_MAJOR = None
if HAS_NVIDIA:
    _m = re.search(r"CUDA Version:\s*(\d+)", _sh("nvidia-smi").stdout)
    CUDA_MAJOR = int(_m.group(1)) if _m else 12
    print(f"✅ GPU detected: {_q.stdout.strip()}  |  driver CUDA {CUDA_MAJOR}")
else:
    print("⚠️  No GPU detected — the notebook will run on CPU.")
    print("    For drug-sized molecules use Runtime ▸ Change runtime type ▸ GPU (T4 / L4 / A100).")

_core = ["pyscf>=2.8", "geometric", "pyscf-dispersion", "tblite", "ase", "rdkit",
         "py3Dmol", "scikit-image", "reportlab", "openpyxl", "psutil"]
print("Installing core packages …")
_r = _sh(f'"{sys.executable}" -m pip install -q ' + " ".join(f'"{p}"' for p in _core))
if _r.returncode != 0:
    print(_r.stdout[-3000:], _r.stderr[-3000:])
    raise RuntimeError("Core installation failed — see the pip log above.")

GPU4PYSCF_INSTALLED = False
if HAS_NVIDIA:
    _pkg = "gpu4pyscf-cuda13x" if (CUDA_MAJOR or 12) >= 13 else "gpu4pyscf-cuda12x"
    print(f"Installing {_pkg} …")
    _r = _sh(f'"{sys.executable}" -m pip install -q {_pkg}')
    if _r.returncode == 0:
        GPU4PYSCF_INSTALLED = True
        if (CUDA_MAJOR or 12) < 13:
            _sh(f'"{sys.executable}" -m pip install -q cutensor-cu12')   # optional speed-up
    else:
        print("⚠️  GPU4PySCF installation failed; continuing on CPU.\n", _r.stderr[-1500:])

print(f"\nSetup finished in {time.time() - _t0:.0f} s")

### 📘 For Your Read — Cell 1.2: Imports, backend detection and shared utilities

**What it does.** Imports everything, decides the compute backend (GPU4PySCF if a working GPU is present, otherwise multithreaded CPU PySCF), sets memory limits, and defines helpers used throughout: figure saving (600 DPI PNG + SVG), table saving, checkpointed stage execution, XYZ I/O, RMSD, and the colour palette.

**How to read the output.** The *Backend* line is the one that matters: **GPU (GPU4PySCF)** or **CPU (PySCF, N threads)**. Every later stage uses it.

**Design notes.** Colours follow one fixed, colour-blind-validated palette across all figures, so the same molecule keeps the same colour from the IR spectra to the comparison plots. Electrostatic and orbital-phase maps use a red ↔ grey ↔ blue diverging scale (red = negative, blue = positive), the convention used in most MEP figures.

In [ ]:
#@title Cell 1.2 — Imports, backend detection & shared utilities
import os, sys, re, io, json, time, math, glob, shutil, zipfile, datetime, platform
import traceback, warnings, contextlib, logging
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm, to_rgb
from IPython.display import display, Markdown, HTML, Image as IPImage
warnings.filterwarnings("ignore")

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Draw, Descriptors, rdMolDescriptors, rdDetermineBonds
from rdkit.Chem.Draw import rdMolDraw2D
RDLogger.DisableLog("rdApp.*")

import pyscf
from pyscf import gto, dft, scf, lib, df
from pyscf.hessian import thermo
from pyscf.geomopt import geometric_solver
from pyscf.tools import cubegen, molden
import geometric
for _name in list(logging.root.manager.loggerDict):
    if _name.startswith("geometric"):
        logging.getLogger(_name).setLevel(logging.ERROR)
logging.getLogger("geometric").setLevel(logging.ERROR)

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    colab_files, IN_COLAB = None, False

# ---------------------------------------------------------------- constants
HARTREE2EV   = 27.211386245988
HARTREE2KCAL = 627.5094740631
BOHR2ANG     = 0.529177210903
ANG2BOHR     = 1.0 / BOHR2ANG
EV2NM        = 1239.841984
KB_HARTREE   = 3.166811563e-6           # Boltzmann constant, Eh/K

# ---------------------------------------------------------------- backend
GPU_OK, GPU_NAME, GPU_MEM_GB, GPU_ERR = False, None, 0.0, ""
try:
    import cupy
    import gpu4pyscf
    if cupy.cuda.runtime.getDeviceCount() > 0:
        _p = cupy.cuda.runtime.getDeviceProperties(0)
        GPU_NAME = _p["name"].decode() if isinstance(_p["name"], bytes) else str(_p["name"])
        GPU_MEM_GB = _p["totalGlobalMem"] / 1e9
        GPU_OK = True
except Exception as _e:
    GPU_ERR = repr(_e)

NCPU = os.cpu_count() or 1
lib.num_threads(NCPU)
try:
    import psutil
    RAM_GB = psutil.virtual_memory().total / 1e9
except Exception:
    RAM_GB = 12.0
MAX_MEMORY_MB = int(RAM_GB * 1000 * 0.60)

def to_np(x):
    """Return a NumPy array whether x lives on GPU or CPU.
    Uses isinstance, so CuPy subclasses (e.g. GPU4PySCF's tagged density matrices) are handled too."""
    if x is None:
        return None
    if GPU_OK:
        import cupy
        if isinstance(x, cupy.ndarray):
            return cupy.asnumpy(x)
    if isinstance(x, (list, tuple)):
        return np.asarray([to_np(v) for v in x])
    if hasattr(x, "get") and not isinstance(x, (np.ndarray, dict)):
        return np.asarray(x.get())
    return np.asarray(x)

def is_gpu_obj(obj):
    return type(obj).__module__.startswith("gpu4pyscf")

def free_gpu():
    """Return cached GPU memory to the driver. CuPy keeps freed blocks in its pool, which can starve
    GPU4PySCF kernels that allocate outside the pool and makes them fail with CUDARuntimeError."""
    import gc
    gc.collect()
    if GPU_OK:
        import cupy
        cupy.get_default_memory_pool().free_all_blocks()
        cupy.get_default_pinned_memory_pool().free_all_blocks()

def gpu_healthy():
    """False if a previous CUDA error has left the GPU context unusable."""
    if not GPU_OK:
        return False
    try:
        import cupy
        return float((cupy.ones(16) * 2).sum().get()) == 32.0
    except Exception:
        return False

# ---------------------------------------------------------------- palette & figure style
PAL   = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
INK, INK2, MUTED, GRIDC, SURF = "#0b0b0b", "#52514e", "#8a8984", "#e4e3df", "#ffffff"
DIVERGING = LinearSegmentedColormap.from_list("binfolab_div", ["#e34948", "#f0efec", "#2a78d6"])
SEQ_BLUE  = LinearSegmentedColormap.from_list("binfolab_seq", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
ELEMENT_COLORS = {"H": "#d9d9d9", "C": "#3a3a3a", "N": "#2a78d6", "O": "#e34948", "S": "#eda100",
                  "P": "#eb6834", "F": "#1baf7a", "Cl": "#008300", "Br": "#8b3a1a", "I": "#6250d6"}
FIG_DPI = 600
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": FIG_DPI, "font.family": "DejaVu Sans", "font.size": 9,
    "axes.titlesize": 10, "axes.titleweight": "bold", "axes.labelsize": 9,
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
    "grid.color": GRIDC, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "lines.linewidth": 2.0, "legend.frameon": False, "axes.prop_cycle": matplotlib.cycler(color=PAL),
})

def mol_color(i):
    """Fixed colour per molecule (input order); molecules beyond the 8th are drawn in grey."""
    return PAL[i] if i < len(PAL) else MUTED

def fmt_time(s):
    s = float(s)
    return f"{s:.0f} s" if s < 90 else (f"{s/60:.1f} min" if s < 5400 else f"{s/3600:.1f} h")

def safe_name(s):
    return re.sub(r"[^A-Za-z0-9_\-]+", "_", str(s)).strip("_")[:40] or "mol"

# ---------------------------------------------------------------- figures & tables (paths set in Part 2)
FIGURES = []        # registry used by the PDF report

def save_fig(fig, stem, caption, part, vector=True):
    path = os.path.join(FIG_DIR, f"{stem}.png")
    fig.savefig(path, dpi=FIG_DPI, bbox_inches="tight", facecolor="white")
    if vector:
        fig.savefig(path[:-4] + ".svg", bbox_inches="tight", facecolor="white")
    FIGURES[:] = [f for f in FIGURES if f["file"] != path] + [{"file": path, "caption": caption, "part": part}]
    with open(os.path.join(OUT_DIR, "figures.json"), "w") as fh:
        json.dump(FIGURES, fh, indent=1)
    plt.show()
    plt.close(fig)
    return path

def save_table(df_, stem, show=True, index=False):
    path = os.path.join(TAB_DIR, f"{stem}.csv")
    df_.to_csv(path, index=index)
    if show:
        display(df_)
    return path

def save_png_bytes(img, path):
    """RDKit drawing helpers return PIL images or IPython images depending on context."""
    if hasattr(img, "save"):
        img.save(path)
    elif hasattr(img, "data"):
        with open(path, "wb") as fh:
            fh.write(img.data)
    else:
        with open(path, "wb") as fh:
            fh.write(img)

# ---------------------------------------------------------------- geometry helpers
def write_xyz(path, symbols, coords_ang, comment=""):
    with open(path, "w") as fh:
        fh.write(f"{len(symbols)}\n{comment}\n")
        for s, (x, y, z) in zip(symbols, coords_ang):
            fh.write(f"{s:<3s} {x:14.8f} {y:14.8f} {z:14.8f}\n")

def read_xyz(path):
    lines = open(path).read().strip().splitlines()
    n = int(lines[0])
    syms, xyz = [], []
    for ln in lines[2:2 + n]:
        p = ln.split()
        syms.append(p[0]); xyz.append([float(v) for v in p[1:4]])
    return syms, np.array(xyz)

def xyz_block(symbols, coords_ang, comment=""):
    s = io.StringIO()
    s.write(f"{len(symbols)}\n{comment}\n")
    for a, (x, y, z) in zip(symbols, coords_ang):
        s.write(f"{a} {x:.6f} {y:.6f} {z:.6f}\n")
    return s.getvalue()

def kabsch_rmsd(A, B):
    A = A - A.mean(0); B = B - B.mean(0)
    U, S, Vt = np.linalg.svd(A.T @ B)
    d = np.sign(np.linalg.det(U @ Vt))
    D = np.diag([1, 1, d])
    R = U @ D @ Vt
    return float(np.sqrt(((A @ R - B) ** 2).sum() / len(A)))

def principal_frame(coords):
    """Rotation that puts the largest molecular extent along x and the second along y."""
    c = coords - coords.mean(0)
    w, v = np.linalg.eigh(np.cov(c.T))
    R = v[:, ::-1]
    if np.linalg.det(R) < 0:
        R[:, 2] *= -1
    return R

def json_default(o):
    if isinstance(o, (np.integer,)):  return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, np.ndarray):     return o.tolist()
    if isinstance(o, complex):        return [o.real, o.imag]
    return str(o)

# ---------------------------------------------------------------- checkpointed stage runner
RES = {}            # per-molecule results (persisted to state.json)
VALIDATION = {}     # benchmark results

def save_state():
    tmp = os.path.join(OUT_DIR, "state.json.tmp")
    with open(tmp, "w") as fh:
        json.dump({"fingerprint": CFG_FINGERPRINT, "molecules": RES, "validation": VALIDATION},
                  fh, default=json_default)
    os.replace(tmp, os.path.join(OUT_DIR, "state.json"))

def run_stage(stage, func, requires=()):
    """Run func(rec) for every molecule, skipping finished work and isolating failures."""
    t_all = time.time()
    for rec in MOLS:
        n = rec["name"]
        r = RES.setdefault(n, {})
        if r.get(stage, {}).get("done") and stage not in RERUN:
            print(f"  ✓ {n:<22s} {stage}: already done (checkpoint) — skipped")
            continue
        missing = [q for q in requires if not r.get(q, {}).get("done")]
        if missing:
            print(f"  – {n:<22s} {stage}: skipped (prerequisite {', '.join(missing)} not available)")
            continue
        t = time.time()
        try:
            out = func(rec) or {}
            out.update(done=True, seconds=round(time.time() - t, 1))
            r[stage] = out
            print(f"  ✓ {n:<22s} {stage}: finished in {fmt_time(time.time() - t)}")
        except Exception as e:
            r[stage] = {"done": False, "error": f"{type(e).__name__}: {e}"}
            r.setdefault("errors", {})[stage] = traceback.format_exc()
            print(f"  ✗ {n:<22s} {stage}: FAILED — {type(e).__name__}: {e}")
        save_state()
        free_gpu()
    print(f"Stage '{stage}' complete in {fmt_time(time.time() - t_all)}")

def ckpt(rec, fname=""):
    d = os.path.join(CKPT_DIR, rec["name"])
    os.makedirs(d, exist_ok=True)
    return os.path.join(d, fname) if fname else d

BACKEND = (f"GPU (GPU4PySCF {gpu4pyscf.__version__}) on {GPU_NAME}, {GPU_MEM_GB:.0f} GB"
           if GPU_OK else f"CPU (PySCF, {NCPU} threads)")
_versions = {"Python": platform.python_version(), "PySCF": pyscf.__version__,
             "geomeTRIC": getattr(geometric, "__version__", "?"),
             "RDKit": Chem.rdBase.rdkitVersion, "NumPy": np.__version__}
if GPU_OK:
    _versions["GPU4PySCF"] = gpu4pyscf.__version__
print("Backend :", BACKEND)
if not GPU_OK and GPU_ERR:
    print("          (GPU unavailable:", GPU_ERR[:120], ")")
print(f"Memory  : {RAM_GB:.1f} GB RAM, PySCF limit {MAX_MEMORY_MB} MB")
display(pd.DataFrame([_versions]).T.rename(columns={0: "version"}))

---
## Part 2 — Settings
Edit the form below, then run Cells 2.1 and 2.2. Changing the level of theory later automatically starts a fresh result set (the old state file is archived), so checkpoints computed at one level are never mixed with another.

### 📘 For Your Read — Cell 2.1: Level of theory and workflow switches

**What to set.**
- **Level preset.** *Standard (publication)* = B3LYP-D3(BJ)/def2-SVP optimisation and frequencies, then B3LYP-D3(BJ)/def2-TZVP single point: the most widely reported protocol for drug-like molecules. *Fast screening* keeps def2-SVP throughout for many molecules. *High accuracy* uses PBE0-D3(BJ)/def2-TZVP geometries and ωB97X-V/def2-TZVPP energies; budget a GPU (preferably L4/A100). Choose *Custom* to use the individual fields.
- **Dispersion = auto** applies the recommended correction per functional: D3(BJ) for B3LYP/PBE0/CAM-B3LYP, D3(0) for M06-2X, none for ωB97X-V (it already contains VV10 non-local correlation).
- **Solvent** switches on IEF-PCM in every DFT step and ALPB in the xTB pre-optimisation. Use *none* to compare with gas-phase literature; use *water* or *DMSO* when interpreting biological or assay conditions.
- **TD-DFT functional.** CAM-B3LYP is the default because global hybrids such as B3LYP systematically underestimate charge-transfer excitations.
- **Frequency scale factor.** Harmonic DFT frequencies run a few percent above experimental fundamentals. Leave 1.0 to report raw harmonic values, or enter the factor for your exact functional/basis from the NIST CCCBDB scaling table.

**Red flags.** def2-TZVP optimisations or many TD-DFT states on CPU will be slow; Cell 3.2 warns you per molecule.

In [ ]:
#@title Cell 2.1 — Settings { display-mode: "form" }
#@markdown ### Project
PROJECT_NAME = "DFTFlow_run1"  #@param {type:"string"}
SAVE_TO_DRIVE = False  #@param {type:"boolean"}
#@markdown ### Level of theory
LEVEL_PRESET = "Standard (publication)"  #@param ["Fast screening", "Standard (publication)", "High accuracy", "Custom"]
#@markdown *Used only when the preset is **Custom**:*
OPT_FUNCTIONAL = "B3LYP"  #@param ["B3LYP", "PBE0", "M06-2X", "CAM-B3LYP"]
OPT_BASIS = "def2-SVP"  #@param ["def2-SVP", "def2-TZVP", "6-31G(d)", "6-311++G(d,p)"]
SP_FUNCTIONAL = "B3LYP"  #@param ["B3LYP", "PBE0", "M06-2X", "CAM-B3LYP", "wB97X-V"]
SP_BASIS = "def2-TZVP"  #@param ["def2-SVP", "def2-TZVP", "def2-TZVPP", "6-311++G(d,p)"]
#@markdown *Always applied:*
DISPERSION = "auto"  #@param ["auto", "D3(BJ)", "D3(0)", "D4", "none"]
SOLVENT = "DMSO"  #@param ["none", "water", "DMSO", "methanol", "ethanol", "acetonitrile", "chloroform", "n-octanol"]
USE_DENSITY_FITTING = True  #@param {type:"boolean"}
#@markdown ### Conformer search
N_CONFORMERS = 30  #@param {type:"slider", min:1, max:200, step:1}
XTB_REFINE_TOP = 5  #@param {type:"slider", min:1, max:20, step:1}
#@markdown ### Analyses
RUN_VALIDATION = True  #@param {type:"boolean"}
RUN_FREQUENCIES = True  #@param {type:"boolean"}
AUTO_FIX_IMAGINARY = True  #@param {type:"boolean"}
FREQ_SCALE_FACTOR = 1.0  #@param {type:"number"}
TEMPERATURE_K = 298.15  #@param {type:"number"}
RUN_FUKUI = True  #@param {type:"boolean"}
RUN_SURFACES = True  #@param {type:"boolean"}
CUBE_POINTS = 64  #@param {type:"slider", min:40, max:120, step:4}
RUN_TDDFT = True  #@param {type:"boolean"}
TD_FUNCTIONAL = "CAM-B3LYP"  #@param ["same as single point", "CAM-B3LYP", "wB97X-V", "B3LYP", "PBE0", "M06-2X"]
TD_NSTATES = 10  #@param {type:"slider", min:3, max:40, step:1}
TD_USE_TDA = False  #@param {type:"boolean"}
#@markdown ### Advanced
FORCE_CPU = False  #@param {type:"boolean"}
OPT_MAX_STEPS = 150  #@param {type:"integer"}
MAX_SCF_CYCLES = 100  #@param {type:"integer"}
RERUN_STAGES = ""  #@param {type:"string"}
#@markdown *RERUN_STAGES: comma-separated stage names to recompute even if checkpointed (conformers, opt, freq, sp, fukui, surfaces, tddft).*

### 📘 For Your Read — Cell 2.2: Resolve settings, workspace and resume

**What it does.** Converts the form into one configuration (functional, basis, dispersion, grids, solvent dielectric), prints the full level-of-theory string in the notation journals expect (`single-point level // optimisation level`), creates the output folders, and loads any previous `state.json` so finished work is not repeated.

**How to read the output.** Copy the *Level of theory* line into your manuscript. If you see *"Resuming: N molecules with saved results"*, earlier work was found and will be reused.

**Output folders.** `figures/`, `tables/`, `structures/`, `cubes/`, `molden/`, `checkpoints/`, `report/`, all under the project folder (on Drive if selected).

In [ ]:
#@title Cell 2.2 — Resolve settings, workspace & resume
PRESETS = {
    "Fast screening":         dict(opt_func="B3LYP", opt_basis="def2-SVP",  sp_func="B3LYP",   sp_basis="def2-SVP"),
    "Standard (publication)": dict(opt_func="B3LYP", opt_basis="def2-SVP",  sp_func="B3LYP",   sp_basis="def2-TZVP"),
    "High accuracy":          dict(opt_func="PBE0",  opt_basis="def2-TZVP", sp_func="wB97X-V", sp_basis="def2-TZVPP"),
}
lvl = PRESETS.get(LEVEL_PRESET, dict(opt_func=OPT_FUNCTIONAL, opt_basis=OPT_BASIS,
                                     sp_func=SP_FUNCTIONAL, sp_basis=SP_BASIS))
XC_PYSCF    = {"B3LYP": "b3lyp", "PBE0": "pbe0", "M06-2X": "m062x", "CAM-B3LYP": "camb3lyp", "wB97X-V": "wb97x-v"}
BASIS_PYSCF = {"def2-SVP": "def2-svp", "def2-TZVP": "def2-tzvp", "def2-TZVPP": "def2-tzvpp",
               "6-31G(d)": "6-31g*", "6-311++G(d,p)": "6-311++g**"}
AUTO_DISP   = {"B3LYP": "D3(BJ)", "PBE0": "D3(BJ)", "CAM-B3LYP": "D3(BJ)", "M06-2X": "D3(0)", "wB97X-V": "none"}
DISP_PYSCF  = {"D3(BJ)": "d3bj", "D3(0)": "d3zero", "D4": "d4", "none": None}
SOLVENT_EPS = {"water": 78.3553, "DMSO": 46.826, "methanol": 32.613, "ethanol": 24.852,
               "acetonitrile": 35.688, "chloroform": 4.7113, "n-octanol": 9.8629}
ALPB_NAME   = {"water": "water", "DMSO": "dmso", "methanol": "methanol", "ethanol": "ethanol",
               "acetonitrile": "acetonitrile", "chloroform": "chloroform", "n-octanol": "octanol"}

def disp_label(func):
    d = AUTO_DISP[func] if DISPERSION == "auto" else DISPERSION
    return "none" if func == "wB97X-V" else d

def lot(func, basis):
    d = disp_label(func)
    return f"{func}{'' if d == 'none' else '-' + d}/{basis}"

TD_FUNC = lvl["sp_func"] if TD_FUNCTIONAL == "same as single point" else TD_FUNCTIONAL
USE_GPU = GPU_OK and not FORCE_CPU
RERUN = {s.strip() for s in RERUN_STAGES.split(",") if s.strip()}
SOLV_TXT = "gas phase" if SOLVENT == "none" else f"IEF-PCM ({SOLVENT}, ε = {SOLVENT_EPS[SOLVENT]})"
LEVEL_STRING = f"{lot(lvl['sp_func'], lvl['sp_basis'])}//{lot(lvl['opt_func'], lvl['opt_basis'])}, {SOLV_TXT}"

CFG = dict(project=PROJECT_NAME, preset=LEVEL_PRESET, **lvl,
           opt_disp=disp_label(lvl["opt_func"]), sp_disp=disp_label(lvl["sp_func"]),
           td_func=TD_FUNC, td_nstates=TD_NSTATES, td_tda=TD_USE_TDA, solvent=SOLVENT,
           density_fitting=USE_DENSITY_FITTING, n_conformers=N_CONFORMERS, xtb_top=XTB_REFINE_TOP,
           freq_scale=FREQ_SCALE_FACTOR, temperature=TEMPERATURE_K, cube_points=CUBE_POINTS,
           backend=BACKEND, created=datetime.datetime.now().isoformat(timespec="seconds"))
# Only settings that change computed numbers enter the fingerprint
CFG_FINGERPRINT = "|".join(str(CFG[k]) for k in ("opt_func", "opt_basis", "sp_func", "sp_basis", "opt_disp",
                                                  "sp_disp", "solvent", "density_fitting"))

# ---------------------------------------------------------------- workspace
_base = "/content" if os.path.isdir("/content") else os.getcwd()
if SAVE_TO_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    _base = "/content/drive/MyDrive/BinFoLab_DFTFlow"
OUT_DIR = os.path.join(_base, safe_name(PROJECT_NAME))
FIG_DIR, TAB_DIR, STRUCT_DIR = (os.path.join(OUT_DIR, d) for d in ("figures", "tables", "structures"))
CUBE_DIR, MOLDEN_DIR, CKPT_DIR, REPORT_DIR = (os.path.join(OUT_DIR, d) for d in ("cubes", "molden", "checkpoints", "report"))
for _d in (OUT_DIR, FIG_DIR, TAB_DIR, STRUCT_DIR, CUBE_DIR, MOLDEN_DIR, CKPT_DIR, REPORT_DIR):
    os.makedirs(_d, exist_ok=True)

# ---------------------------------------------------------------- resume
_state = os.path.join(OUT_DIR, "state.json")
RES.clear(); VALIDATION.clear(); FIGURES.clear()
if os.path.exists(_state):
    _old = json.load(open(_state))
    if _old.get("fingerprint") == CFG_FINGERPRINT:
        RES.update(_old.get("molecules", {})); VALIDATION.update(_old.get("validation", {}))
        if os.path.exists(os.path.join(OUT_DIR, "figures.json")):
            FIGURES.extend(json.load(open(os.path.join(OUT_DIR, "figures.json"))))
        print(f"🔁 Resuming: {len(RES)} molecule(s) with saved results in {OUT_DIR}")
    else:
        _arch = _state.replace(".json", f"_archived_{datetime.datetime.now():%Y%m%d_%H%M%S}.json")
        os.replace(_state, _arch)
        shutil.rmtree(CKPT_DIR); os.makedirs(CKPT_DIR)
        print("⚠️  Level of theory changed since the last run: previous results archived to", os.path.basename(_arch))
json.dump(CFG, open(os.path.join(OUT_DIR, "config.json"), "w"), indent=1)

display(Markdown(f"""
| Setting | Value |
|---|---|
| **Level of theory** | `{LEVEL_STRING}` |
| Geometry / frequencies | {lot(lvl['opt_func'], lvl['opt_basis'])} |
| Single point / properties | {lot(lvl['sp_func'], lvl['sp_basis'])} |
| TD-DFT | {TD_FUNC}/{lvl['sp_basis']}, {TD_NSTATES} states, {'TDA' if TD_USE_TDA else 'full TD-DFT (RPA)'} |
| Solvent | {SOLV_TXT} |
| Density fitting (RI-J/K) | {'on (def2-universal-JKFIT)' if USE_DENSITY_FITTING else 'off'} |
| Backend | {'GPU4PySCF' if USE_GPU else 'CPU PySCF'} |
| Output folder | `{OUT_DIR}` |
"""))

---
## Part 3 — Input Molecules

### 📘 For Your Read — Cell 3.1: Provide molecules

**Three ways to provide molecules.**
1. **Demo set**: paracetamol, aspirin and caffeine, to see the whole pipeline work end to end.
2. **Paste table**: edit `MOLECULE_TABLE` in the cell (CSV columns `name, smiles, charge, multiplicity`; charge and multiplicity may be left blank and are then derived from the SMILES).
3. **Upload file**: `.csv`, `.smi`, `.sdf`, `.mol` or `.xyz`. SDF/MOL/XYZ files keep their 3D coordinates; tick **USE_INPUT_3D** to start from them instead of running the conformer search, which is useful for docked poses or crystal structures. For XYZ files, bonds are perceived automatically using `XYZ_CHARGE`.

A template file `sample_input_template.csv` is written to the project folder so the expected format is unambiguous.

**Tips.** Give each molecule a short name; it becomes the file prefix for every output. Use the protonation state you want to study (e.g. `[NH3+]` for an amine at pH 7), because DFT will not change it for you.

In [ ]:
#@title Cell 3.1 — Provide molecules
INPUT_MODE = "Paste table"  #@param ["Demo set", "Paste table", "Upload file"]
USE_INPUT_3D = False  #@param {type:"boolean"}
XYZ_CHARGE = 0  #@param {type:"integer"}
UPLOAD_PATHS = ""  #@param {type:"string"}
#@markdown *UPLOAD_PATHS (optional): comma-separated paths of files already in the runtime (e.g. on Drive); used instead of the upload dialog.*

# ---- edit this table when INPUT_MODE = "Paste table" ------------------------------
MOLECULE_TABLE = """name,smiles,charge,multiplicity
paracetamol,CC(=O)Nc1ccc(O)cc1,0,1
aspirin,CC(=O)Oc1ccccc1C(=O)O,0,1
"""
# -----------------------------------------------------------------------------------
DEMO_TABLE = """name,smiles,charge,multiplicity
paracetamol,CC(=O)Nc1ccc(O)cc1,0,1
aspirin,CC(=O)Oc1ccccc1C(=O)O,0,1
caffeine,Cn1c(=O)c2c(ncn2C)n(C)c1=O,0,1
"""
TEMPLATE = """name,smiles,charge,multiplicity
paracetamol,CC(=O)Nc1ccc(O)cc1,0,1
acetate_anion,CC(=O)[O-],-1,1
quercetin,O=c1c(O)c(-c2ccc(O)c(O)c2)oc2cc(O)cc(O)c12,,
"""
with open(os.path.join(OUT_DIR, "sample_input_template.csv"), "w") as fh:
    fh.write(TEMPLATE)

def _rows_from_csv_text(txt, source):
    df_ = pd.read_csv(io.StringIO(txt)) if isinstance(txt, str) else txt
    cols = {c.lower().strip(): c for c in df_.columns}
    smi_c = cols.get("smiles") or cols.get("smi")
    if smi_c is None:
        raise ValueError(f"{source}: no 'smiles' column found (columns: {list(df_.columns)})")
    name_c = cols.get("name") or cols.get("id") or cols.get("title")
    ch_c, mu_c = cols.get("charge"), cols.get("multiplicity") or cols.get("mult")
    rows = []
    for i, row in df_.iterrows():
        rows.append(dict(name=str(row[name_c]) if name_c else f"mol{i+1}", smiles=str(row[smi_c]).strip(),
                         charge=row[ch_c] if ch_c else None, mult=row[mu_c] if mu_c else None,
                         molblock=None, source=source))
    return rows

def _rows_from_file(path):
    ext = os.path.splitext(path)[1].lower()
    base = os.path.splitext(os.path.basename(path))[0]
    if ext in (".csv", ".tsv"):
        return _rows_from_csv_text(pd.read_csv(path, sep="\t" if ext == ".tsv" else ","), base)
    if ext in (".smi", ".txt"):
        rows = []
        for i, ln in enumerate(open(path).read().splitlines()):
            p = ln.split()
            if p and not p[0].startswith("#"):
                rows.append(dict(name=p[1] if len(p) > 1 else f"{base}_{i+1}", smiles=p[0], charge=None,
                                 mult=None, molblock=None, source=base))
        return rows
    if ext in (".sdf", ".mol"):
        rows = []
        for i, m in enumerate(Chem.SDMolSupplier(path, removeHs=False)):
            if m is None:
                print(f"  ⚠️ {base}: record {i+1} could not be parsed — skipped"); continue
            nm = m.GetProp("_Name").strip() if m.HasProp("_Name") and m.GetProp("_Name").strip() else f"{base}_{i+1}"
            mult = m.GetProp("multiplicity") if m.HasProp("multiplicity") else None
            rows.append(dict(name=nm, smiles=None, charge=None, mult=mult, molblock=Chem.MolToMolBlock(m), source=base))
        return rows
    if ext == ".xyz":
        m = Chem.MolFromXYZFile(path)
        rdDetermineBonds.DetermineBonds(m, charge=int(XYZ_CHARGE))
        return [dict(name=base, smiles=None, charge=int(XYZ_CHARGE), mult=None,
                     molblock=Chem.MolToMolBlock(m), source=base)]
    raise ValueError(f"Unsupported file type: {ext}")

RAW_INPUT = []
if INPUT_MODE == "Demo set":
    RAW_INPUT = _rows_from_csv_text(DEMO_TABLE, "demo")
elif INPUT_MODE == "Paste table":
    RAW_INPUT = _rows_from_csv_text(MOLECULE_TABLE, "pasted table")
else:
    paths = [p.strip() for p in UPLOAD_PATHS.split(",") if p.strip()]
    if not paths and IN_COLAB:
        up = colab_files.upload()
        paths = list(up.keys())
    if not paths:
        raise RuntimeError("No file provided. Upload a file or set UPLOAD_PATHS.")
    for p in paths:
        RAW_INPUT += _rows_from_file(p)
print(f"Read {len(RAW_INPUT)} input record(s) from: {INPUT_MODE}")
print("Format template written to:", os.path.join(OUT_DIR, "sample_input_template.csv"))

### 📘 For Your Read — Cell 3.2: Parse, validate and estimate cost

**What it does.** For each molecule it builds an RDKit structure with explicit hydrogens, then checks:
- **Charge.** Taken from the formal charges in the SMILES. If your table says something different, the SMILES wins and a warning is printed.
- **Multiplicity and electron parity.** An even electron count needs an odd multiplicity (singlet, triplet), and vice versa. Impossible combinations are rejected.
- **Basis-set coverage.** Every element must exist in both chosen basis sets. def2 covers H–Rn; the Pople sets stop around Kr.
- **Cost.** The number of basis functions at the single-point level is the best quick proxy for DFT cost. The *tier* column is a rough guide, not a prediction.

**How to read the output.** A table with one row per molecule and a *status* column; only rows marked **ok** continue. The 2D grid confirms that the structures are the ones you intended.

**Red flags.** *Heavy* or *Very heavy* tiers on a CPU runtime; consider the Fast preset or a GPU.

In [ ]:
#@title Cell 3.2 — Parse, validate & estimate cost
def _cost_tier(nao):
    if nao <= 350:  return "Light"
    if nao <= 700:  return "Moderate"
    if nao <= 1100: return "Heavy (GPU recommended)"
    return "Very heavy (Fast preset / A100)"

MOLS, rows, _seen = [], [], set()
for raw in RAW_INPUT:
    status, note = "ok", ""
    try:
        if raw["molblock"]:
            m = Chem.MolFromMolBlock(raw["molblock"], removeHs=False)
            has3d = m is not None and m.GetNumConformers() > 0 and m.GetConformer().Is3D()
            if m is not None and has3d and not any(a.GetAtomicNum() == 1 for a in m.GetAtoms()):
                m = Chem.AddHs(m, addCoords=True)
            elif m is not None:
                m = Chem.AddHs(m, addCoords=has3d)
            smi = Chem.MolToSmiles(Chem.RemoveHs(m)) if m is not None else ""
        else:
            m0 = Chem.MolFromSmiles(raw["smiles"])
            m = Chem.AddHs(m0) if m0 is not None else None
            has3d, smi = False, raw["smiles"]
        if m is None:
            raise ValueError("RDKit could not parse the structure")
        frags = Chem.GetMolFrags(m)
        if len(frags) > 1:
            note += "multiple fragments (salt/complex) kept together; "
        q_formal = Chem.GetFormalCharge(m)
        q_user = raw.get("charge")
        q = q_formal
        if q_user is not None and str(q_user).strip() not in ("", "nan", "None") and int(float(q_user)) != q_formal:
            note += f"table charge {int(float(q_user))} ≠ SMILES formal charge {q_formal}; using {q_formal}; "
        mu = raw.get("mult")
        mult = int(float(mu)) if mu is not None and str(mu).strip() not in ("", "nan", "None") else \
               Descriptors.NumRadicalElectrons(m) + 1
        nelec = sum(a.GetAtomicNum() for a in m.GetAtoms()) - q
        if (nelec + mult - 1) % 2 != 0:
            raise ValueError(f"{nelec} electrons cannot have multiplicity {mult}")
        syms = [a.GetSymbol() for a in m.GetAtoms()]
        dummy = [(s, (0.0, 0.0, 2.0 * i)) for i, s in enumerate(syms)]
        naos = {}
        for key in ("opt_basis", "sp_basis"):
            b = BASIS_PYSCF[CFG[key]]
            hv = sorted({s for s in syms if gto.charge(s) > 36})
            tm = gto.M(atom=dummy, basis=b, ecp={s: b for s in hv} if (b.startswith("def2") and hv) else None,
                       charge=q, spin=mult - 1, verbose=0)
            naos[key] = tm.nao
        name = safe_name(raw["name"])
        k = 2
        while name in _seen:
            name = f"{safe_name(raw['name'])}_{k}"; k += 1
        _seen.add(name)
        if USE_INPUT_3D and not has3d:
            note += "no 3D coordinates in input — conformer search will be used; "
        MOLS.append(dict(name=name, smiles=smi, rdmol=m, charge=q, mult=mult, has3d=has3d,
                         symbols=syms, nao_opt=naos["opt_basis"], nao_sp=naos["sp_basis"]))
    except Exception as e:
        status, name, q, mult, syms, naos = "rejected", safe_name(raw.get("name", "?")), None, None, [], {}
        note += str(e)
        m = None
    rows.append({"name": name, "formula": rdMolDescriptors.CalcMolFormula(m) if m is not None else "",
                 "MW": round(Descriptors.MolWt(m), 2) if m is not None else None,
                 "heavy atoms": m.GetNumHeavyAtoms() if m is not None else None,
                 "atoms": m.GetNumAtoms() if m is not None else None,
                 "charge": q, "mult": mult,
                 "basis fns (opt)": naos.get("opt_basis"), "basis fns (SP)": naos.get("sp_basis"),
                 "tier": _cost_tier(naos["sp_basis"]) if naos else "", "status": status, "notes": note.strip("; ")})

INPUT_TABLE = pd.DataFrame(rows)
save_table(INPUT_TABLE, "01_input_validation")
if not MOLS:
    raise RuntimeError("No valid molecules — fix the input and re-run Cell 3.1.")
if not USE_GPU and max(r["nao_sp"] for r in MOLS) > 700:
    print("⚠️  CPU runtime with large molecules: expect long run times. A GPU runtime is strongly recommended.")

# ---- 2D overview grid
_ms = [Chem.RemoveHs(r["rdmol"]) for r in MOLS]
for _m in _ms:
    AllChem.Compute2DCoords(_m)
_img = Draw.MolsToGridImage(_ms, molsPerRow=min(4, len(_ms)), subImgSize=(420, 340),
                            legends=[f"{r['name']}  (q={r['charge']}, 2S+1={r['mult']})" for r in MOLS])
_p = os.path.join(FIG_DIR, "01_input_structures.png")
save_png_bytes(_img, _p)
FIGURES[:] = [f for f in FIGURES if f["file"] != _p] + [{"file": _p, "caption": "Input structures (2D depiction).", "part": 3}]
json.dump(FIGURES, open(os.path.join(OUT_DIR, "figures.json"), "w"), indent=1)
display(IPImage(_p, width=min(900, 230 * min(4, len(_ms)))))
print(f"{len(MOLS)} molecule(s) accepted for the DFT workflow.")

---
## Part 4 — DFT Engine Core
These cells define the machinery that every later stage uses. Read them once to understand what is being computed; you do not need to edit them.

### 📘 For Your Read — Cell 4.1: Molecule and Kohn–Sham builders, robust SCF

**What it does.** Defines how a calculation is set up:
- `build_mole` creates the PySCF molecule (Ångström input, def2 effective-core potentials applied automatically for elements beyond Kr).
- `make_mf` creates a restricted (closed-shell) or unrestricted (open-shell) Kohn–Sham object on GPU or CPU with the chosen functional, empirical dispersion, integration grid (level 3; level 4 for M06-2X, whose kinetic-energy-density terms are grid-sensitive), optional RI density fitting, and optional IEF-PCM solvation.
- `run_scf` runs the SCF to a tight 10⁻⁹ Eh threshold. If it fails, it retries with level shifting and damping and then (on CPU) a second-order Newton solver, before giving up with a clear error.

**Why it matters.** Unconverged SCFs are the most common silent error in DFT pipelines. Here a molecule either converges or is flagged; it never passes through with a half-converged density.

In [ ]:
#@title Cell 4.1 — Molecule & Kohn–Sham builders, robust SCF
def build_mole(symbols, coords_ang, charge, mult, basis_label, verbose=0):
    b = BASIS_PYSCF[basis_label]
    mol = gto.Mole()
    mol.atom = [(s, tuple(map(float, c))) for s, c in zip(symbols, coords_ang)]
    mol.basis = b
    heavy = sorted({s for s in symbols if gto.charge(s) > 36})     # def2 ECPs exist only beyond Kr
    if b.startswith("def2") and heavy:
        mol.ecp = {s: b for s in heavy}
    mol.charge, mol.spin = int(charge), int(mult) - 1
    mol.verbose, mol.max_memory = verbose, MAX_MEMORY_MB
    mol.build()
    return mol

def make_mf(mol, functional, solvent=True, gpu=None, density_fit=None):
    gpu = USE_GPU if gpu is None else gpu
    density_fit = USE_DENSITY_FITTING if density_fit is None else density_fit
    xc = XC_PYSCF[functional]
    closed = (mol.spin == 0)
    if gpu:
        from gpu4pyscf import dft as gdft
        mf = gdft.RKS(mol, xc=xc) if closed else gdft.UKS(mol, xc=xc)
    else:
        mf = dft.RKS(mol, xc=xc) if closed else dft.UKS(mol, xc=xc)
    if density_fit:
        mf = mf.density_fit()            # must precede PCM
    mf.disp = DISP_PYSCF[disp_label(functional)]
    mf.grids.level = 4 if functional == "M06-2X" else 3
    mf.conv_tol = 1e-9
    mf.max_cycle = MAX_SCF_CYCLES
    mf.verbose = 0
    if solvent and SOLVENT != "none":
        mf = mf.PCM()
        mf.with_solvent.method = "IEF-PCM"
        mf.with_solvent.eps = SOLVENT_EPS[SOLVENT]
    return mf

def run_scf(mf, dm0=None):
    mf.kernel(dm0=dm0) if dm0 is not None else mf.kernel()
    if not mf.converged:
        mf.level_shift, mf.damp, mf.diis_start_cycle = 0.3, 0.5, 2
        mf.max_cycle = 3 * MAX_SCF_CYCLES
        mf.kernel(dm0=mf.make_rdm1())
        mf.level_shift, mf.damp = 0.0, 0.0
    if not mf.converged and not is_gpu_obj(mf):
        try:
            mf2 = mf.newton()
            mf2.kernel(mf.mo_coeff, mf.mo_occ)
            if mf2.converged:
                mf = mf2
        except Exception:
            pass
    if not mf.converged:
        raise RuntimeError("SCF did not converge (tried DIIS, level shift/damping and Newton)")
    return mf

def mf_for(rec, coords_ang, functional, basis, charge=None, mult=None, solvent=True, gpu=None, density_fit=None):
    mol = build_mole(rec["symbols"], coords_ang, rec["charge"] if charge is None else charge,
                     rec["mult"] if mult is None else mult, basis)
    return run_scf(make_mf(mol, functional, solvent=solvent, gpu=gpu, density_fit=density_fit))

print("Engine ready:", "GPU4PySCF" if USE_GPU else "CPU PySCF", "| density fitting", "on" if USE_DENSITY_FITTING else "off")

### 📘 For Your Read — Cell 4.2: Resumable geometry optimiser

**What it does.** Wraps PySCF's interface to **geomeTRIC**, which optimises in translation–rotation-internal coordinates (TRIC) and typically needs far fewer steps than Cartesian optimisers for flexible molecules. Convergence uses geomeTRIC's default criteria, equivalent to Gaussian's default (max force 4.5×10⁻⁴ Eh/bohr, RMS force 3×10⁻⁴, max displacement 1.8×10⁻³ Å, RMS displacement 1.2×10⁻³ Å, ΔE 10⁻⁶ Eh).

**Checkpointing.** After every optimisation cycle the current geometry and the energy/gradient trace are written to `checkpoints/<molecule>/`. If the runtime dies mid-optimisation, the next run restarts from the last geometry instead of from scratch.

In [ ]:
#@title Cell 4.2 — Resumable geometry optimiser
def optimize_geometry(symbols, start_ang, charge, mult, functional, basis, tag, ckpt_dir,
                      solvent=True, maxsteps=None):
    last_xyz = os.path.join(ckpt_dir, f"{tag}_last.xyz")
    trace_f  = os.path.join(ckpt_dir, f"{tag}_trace.json")
    trace = json.load(open(trace_f)) if os.path.exists(last_xyz) and os.path.exists(trace_f) else []
    if os.path.exists(last_xyz):
        _, start_ang = read_xyz(last_xyz)
        print(f"      resuming optimisation from checkpoint (cycle {len(trace)})")
    mol = build_mole(symbols, start_ang, charge, mult, basis)
    mf = make_mf(mol, functional, solvent=solvent)

    def _cb(env):
        g = to_np(env["gradients"]).reshape(-1, 3)
        trace.append({"cycle": len(trace) + 1, "energy": float(env["energy"]),
                      "grad_rms": float(np.sqrt((g ** 2).mean())), "grad_max": float(np.abs(g).max())})
        write_xyz(last_xyz, symbols, env["mol"].atom_coords(unit="Ang"), f"{tag} cycle {len(trace)}")
        json.dump(trace, open(trace_f, "w"))

    with contextlib.redirect_stdout(io.StringIO()):
        conv, mol_eq = geometric_solver.kernel(mf, maxsteps=maxsteps or OPT_MAX_STEPS, callback=_cb)
    coords = mol_eq.atom_coords(unit="Ang")
    mf_eq = run_scf(make_mf(mol_eq, functional, solvent=solvent))
    return dict(converged=bool(conv), coords=coords, energy=float(mf_eq.e_tot), trace=trace, mf=mf_eq)

print("Optimiser ready (geomeTRIC", getattr(geometric, "__version__", ""), ")")

### 📘 For Your Read — Cell 4.3: Backend smoke test

**What it does.** Runs one quick SCF on water with the optimisation functional to prove the backend, functional, dispersion and solvent settings work together, and times it. If the GPU path throws an error, the notebook switches to CPU here rather than failing halfway through the real run.

**What you should see.** *Smoke test passed* with an energy near −76.3 Eh (exact value depends on functional/basis) and a timing.

In [ ]:
#@title Cell 4.3 — Backend smoke test
_w = (["O", "H", "H"], np.array([[0, 0, 0.117], [0, 0.757, -0.469], [0, -0.757, -0.469]]))
def _smoke(gpu):
    t = time.time()
    mol = build_mole(*_w, 0, 1, CFG["opt_basis"])
    mf = run_scf(make_mf(mol, CFG["opt_func"], gpu=gpu))
    return mf.e_tot, time.time() - t
try:
    _e, _t = _smoke(USE_GPU)
except Exception as e:
    if USE_GPU:
        print("⚠️  GPU smoke test failed:", repr(e)[:200], "\n    Falling back to CPU for the whole run.")
        USE_GPU = False
        BACKEND = f"CPU (PySCF, {NCPU} threads; GPU fallback)"
        CFG["backend"] = BACKEND
        _e, _t = _smoke(False)
    else:
        raise
print(f"✅ Smoke test passed on {'GPU' if USE_GPU else 'CPU'}: E(H2O, {lot(CFG['opt_func'], CFG['opt_basis'])}) = {_e:.6f} Eh in {_t:.1f} s")

---
## Part 5 — Method Validation Benchmark

### 📘 For Your Read — Cell 5.1: Benchmark against experimental reference molecules

**What it does.** Before trusting the method on your molecules, it reproduces experimentally known data for three small reference molecules (water, formaldehyde, ammonia): equilibrium bond lengths and angles at your optimisation level, and dipole moments at your single-point level, all in the gas phase to match the experiments. Reference values are standard experimental equilibrium data as tabulated in NIST CCCBDB.

**Pass criteria.** |Δbond| ≤ 0.025 Å, |Δangle| ≤ 2.5°, |Δdipole| ≤ 0.25 D, the typical error range of hybrid DFT with double-/triple-ζ basis sets.

**Interpreting a FAIL.** Large failures (e.g. bonds off by > 0.05 Å) point to a setup problem. Small ones are usually basis-set effects: def2-SVP lengthens X–H bonds by about 0.01–0.02 Å, and dipole moments need a triple-ζ basis (which the Standard preset uses for the single point), so a dipole FAIL with a def2-SVP single point is expected rather than alarming.

**How to read the output.** A table of computed vs experimental values with PASS/FAIL and a deviation chart where the grey band is the tolerance. Reporting this table in your supplementary information shows reviewers the protocol was checked.

In [ ]:
#@title Cell 5.1 — Benchmark against experiment
REFERENCE_SET = {
    "water": dict(symbols=["O", "H", "H"], xyz=[[0, 0, 0.12], [0, 0.76, -0.47], [0, -0.76, -0.47]],
                  bonds={"r(O–H)": ((0, 1), 0.958)}, angles={"∠H–O–H": ((1, 0, 2), 104.5)}, dipole=1.85),
    "formaldehyde": dict(symbols=["C", "O", "H", "H"], xyz=[[0, 0, -0.53], [0, 0, 0.68], [0, 0.94, -1.11], [0, -0.94, -1.11]],
                  bonds={"r(C=O)": ((0, 1), 1.205), "r(C–H)": ((0, 2), 1.101)}, angles={"∠H–C–H": ((2, 0, 3), 116.7)}, dipole=2.33),
    "ammonia": dict(symbols=["N", "H", "H", "H"], xyz=[[0, 0, 0.12], [0, 0.94, -0.27], [0.81, -0.47, -0.27], [-0.81, -0.47, -0.27]],
                  bonds={"r(N–H)": ((0, 1), 1.012)}, angles={"∠H–N–H": ((1, 0, 2), 106.7)}, dipole=1.47),
}
TOL = {"bond": 0.025, "angle": 2.5, "dipole": 0.25}

def _angle(c, i, j, k):
    a, b = c[i] - c[j], c[k] - c[j]
    return math.degrees(math.acos(np.dot(a, b) / np.linalg.norm(a) / np.linalg.norm(b)))

if RUN_VALIDATION:
    vrows = list(VALIDATION.get("rows", []))
    done = {r["molecule"] for r in vrows}
    vdir = os.path.join(CKPT_DIR, "_validation"); os.makedirs(vdir, exist_ok=True)
    for nm, ref in REFERENCE_SET.items():
        if nm in done:
            print(f"  ✓ {nm}: from checkpoint"); continue
        t = time.time()
        o = optimize_geometry(ref["symbols"], np.array(ref["xyz"], float), 0, 1, CFG["opt_func"], CFG["opt_basis"],
                              nm, vdir, solvent=False)
        c = o["coords"]
        mol = build_mole(ref["symbols"], c, 0, 1, CFG["sp_basis"])
        mfs = run_scf(make_mf(mol, CFG["sp_func"], solvent=False))
        mu = float(np.linalg.norm(to_np(mfs.dip_moment(unit="Debye", verbose=0))))
        for lab, ((i, j), ex) in ref["bonds"].items():
            v = float(np.linalg.norm(c[i] - c[j]))
            vrows.append(dict(molecule=nm, quantity=lab, kind="bond", unit="Å", computed=round(v, 4), experiment=ex))
        for lab, ((i, j, k), ex) in ref["angles"].items():
            vrows.append(dict(molecule=nm, quantity=lab, kind="angle", unit="°", computed=round(_angle(c, i, j, k), 2), experiment=ex))
        vrows.append(dict(molecule=nm, quantity="dipole μ", kind="dipole", unit="D", computed=round(mu, 3), experiment=ref["dipole"]))
        print(f"  ✓ {nm}: optimised in {len(o['trace'])} cycles, {fmt_time(time.time() - t)}")
        VALIDATION["rows"] = vrows; save_state()
    VT = pd.DataFrame(vrows)
    VT["deviation"] = (VT["computed"] - VT["experiment"]).round(4)
    VT["tolerance"] = VT["kind"].map(TOL)
    VT["result"] = np.where(VT["deviation"].abs() <= VT["tolerance"], "PASS", "FAIL")
    VALIDATION.update(rows=vrows, n_pass=int((VT["result"] == "PASS").sum()), n_total=len(VT),
                      level=f"{lot(CFG['opt_func'], CFG['opt_basis'])} geometries; {lot(CFG['sp_func'], CFG['sp_basis'])} dipoles")
    save_state()
    save_table(VT[["molecule", "quantity", "unit", "computed", "experiment", "deviation", "tolerance", "result"]],
               "02_validation_benchmark")
    print(f"Validation: {VALIDATION['n_pass']}/{VALIDATION['n_total']} quantities within tolerance.")
else:
    VT = None
    print("Validation skipped (RUN_VALIDATION = False).")

### 📘 For Your Read — Cell 5.2: Validation figure

**How to read it.** Three panels (bond lengths, angles, dipoles). Each bar is computed − experimental; the grey band is the tolerance. Bars inside the band pass. Hybrid functionals with def2-SVP usually give slightly long X–H bonds and slightly small angles; def2-TZVP tightens both.

In [ ]:
#@title Cell 5.2 — Validation figure
if VT is not None and len(VT):
    kinds = [("bond", "Bond length error (Å)"), ("angle", "Angle error (°)"), ("dipole", "Dipole error (D)")]
    fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.2))
    for ax, (k, lab) in zip(axes, kinds):
        sub = VT[VT["kind"] == k].reset_index(drop=True)
        labels = [f"{m}\n{q}" for m, q in zip(sub["molecule"], sub["quantity"])]
        tol = TOL[k]
        ax.axhspan(-tol, tol, color=GRIDC, alpha=0.7, lw=0, label="tolerance")
        ax.bar(range(len(sub)), sub["deviation"], color=PAL[0], width=0.55)
        for i, (d, res) in enumerate(zip(sub["deviation"], sub["result"])):
            ax.annotate(f"{d:+.3f}" if k != "angle" else f"{d:+.1f}", (i, d), xytext=(0, 3 if d >= 0 else -10),
                        textcoords="offset points", ha="center", fontsize=7.5, color=INK2)
        ax.axhline(0, color=INK2, lw=0.8)
        ax.set_xticks(range(len(sub))); ax.set_xticklabels(labels, fontsize=7.5)
        ax.set_title(lab, loc="left")
        lim = max(tol * 1.6, sub["deviation"].abs().max() * 1.35)
        ax.set_ylim(-lim, lim); ax.grid(axis="x", visible=False)
    fig.suptitle(f"Method validation vs experiment — {VALIDATION['level']}", fontsize=10, x=0.01, ha="left")
    fig.tight_layout()
    save_fig(fig, "02_validation_benchmark",
             f"Deviation of computed from experimental structural parameters and dipole moments ({VALIDATION['level']}); shaded band = tolerance.", 5)

---
## Part 6 — Conformer Search and Semi-empirical Pre-optimisation

### 📘 For Your Read — Cell 6.1: Conformer search functions

**What it does.** DFT properties depend on conformation, so a sensible starting structure matters:
1. **ETKDGv3** distance-geometry embedding generates up to `N_CONFORMERS` diverse 3D conformers (RMSD pruning 0.3 Å).
2. Each is relaxed with the **MMFF94** force field (UFF if MMFF lacks parameters).
3. The `XTB_REFINE_TOP` lowest are re-optimised with **GFN2-xTB** (tblite), a quantum-mechanical tight-binding method that ranks conformers far more reliably than force fields; ALPB implicit solvent is applied if a solvent is set.
4. The structure with the lowest xTB energy whose bonding pattern is unchanged goes on to DFT. A connectivity check rejects conformers in which xTB moved a proton or broke a bond.

Relative energies and Boltzmann populations at `TEMPERATURE_K` are recorded, so you can report whether one conformer dominates.

In [ ]:
#@title Cell 6.1 — Conformer search functions
from ase import Atoms
from ase.optimize import BFGS

def _adjacency(symbols, coords, charge):
    m = Chem.RWMol()
    for s in symbols:
        m.AddAtom(Chem.Atom(s))
    conf = Chem.Conformer(len(symbols))
    for i, c in enumerate(coords):
        conf.SetAtomPosition(i, [float(v) for v in c])
    m.AddConformer(conf)
    rdDetermineBonds.DetermineConnectivity(m)
    return Chem.GetAdjacencyMatrix(m)

def xtb_optimize(symbols, coords, charge, mult):
    from tblite.ase import TBLite
    at = Atoms(symbols, positions=coords)
    kw = dict(method="GFN2-xTB", charge=int(charge), multiplicity=int(mult), verbosity=0)
    if SOLVENT != "none":
        kw["solvation"] = ("alpb", ALPB_NAME[SOLVENT])
    at.calc = TBLite(**kw)
    BFGS(at, logfile=None).run(fmax=0.03, steps=500)
    return at.get_potential_energy() / HARTREE2EV, at.get_positions()

def conformer_search(rec):
    m = Chem.Mol(rec["rdmol"])
    syms, q, mult = rec["symbols"], rec["charge"], rec["mult"]
    ref_adj = None
    if USE_INPUT_3D and rec["has3d"]:
        seeds = [(None, m.GetConformer().GetPositions())]
        ref_adj = _adjacency(syms, seeds[0][1], q)
        method = "input 3D geometry"
    else:
        p = AllChem.ETKDGv3(); p.randomSeed = 20261023; p.pruneRmsThresh = 0.3; p.numThreads = 0
        cids = list(AllChem.EmbedMultipleConfs(m, numConfs=N_CONFORMERS, params=p))
        if not cids:
            p.useRandomCoords = True
            cids = list(AllChem.EmbedMultipleConfs(m, numConfs=N_CONFORMERS, params=p))
        if not cids:
            raise RuntimeError("RDKit could not embed a 3D structure")
        if AllChem.MMFFHasAllMoleculeParams(m):
            ff = AllChem.MMFFOptimizeMoleculeConfs(m, maxIters=2000, numThreads=0); method = "ETKDGv3 + MMFF94"
        else:
            ff = AllChem.UFFOptimizeMoleculeConfs(m, maxIters=2000, numThreads=0); method = "ETKDGv3 + UFF"
        e_ff = np.array([e for _, e in ff])
        order = np.argsort(e_ff)
        seeds = [(float(e_ff[i]), m.GetConformer(cids[i]).GetPositions()) for i in order[:XTB_REFINE_TOP]]
        ref_adj = Chem.GetAdjacencyMatrix(m)
    confs = []
    for k, (eff, xyz) in enumerate(seeds):
        try:
            e_x, xyz_x = xtb_optimize(syms, xyz, q, mult)
            intact = np.array_equal(_adjacency(syms, xyz_x, q), ref_adj) if ref_adj is not None else True
            confs.append(dict(idx=k, e_ff=eff, e_xtb=float(e_x), intact=bool(intact), xyz=xyz_x.tolist()))
        except Exception as e:
            confs.append(dict(idx=k, e_ff=eff, e_xtb=None, intact=False, xyz=np.asarray(xyz).tolist(), error=str(e)))
    ok = [c for c in confs if c["e_xtb"] is not None and c["intact"]]
    if not ok:
        print(f"      ⚠️ xTB failed or changed bonding for all conformers — using force-field geometry")
        best = dict(xyz=np.asarray(seeds[0][1]).tolist(), e_xtb=None)
        engine = method
    else:
        e = np.array([c["e_xtb"] for c in ok])
        rel = (e - e.min()) * HARTREE2KCAL
        w = np.exp(-(e - e.min()) / (KB_HARTREE * TEMPERATURE_K)); w /= w.sum()
        for c, r_, w_ in zip(ok, rel, w):
            c["rel_kcal"], c["boltzmann"] = float(r_), float(w_)
        best = ok[int(np.argmin(e))]
        engine = method + " → GFN2-xTB" + (f" (ALPB {SOLVENT})" if SOLVENT != "none" else "")
    xyz_best = np.array(best["xyz"])
    write_xyz(os.path.join(STRUCT_DIR, f"{rec['name']}_xtb.xyz"), syms, xyz_best, f"{rec['name']} {engine}")
    return dict(method=engine, n_embedded=len(seeds) if USE_INPUT_3D and rec["has3d"] else None,
                conformers=[{k: v for k, v in c.items() if k != "xyz"} for c in confs],
                best_xyz=xyz_best.tolist(), best_population=best.get("boltzmann"))

print("Conformer search ready (RDKit", Chem.rdBase.rdkitVersion, "+ tblite GFN2-xTB)")

### 📘 For Your Read — Cell 6.2: Run the conformer search

**What you should see.** One line per molecule. Rigid molecules finish in seconds; flexible ones (many rotatable bonds) take longer because each refined conformer is a full xTB optimisation.

**How to read the table.** *Pop. of best* is the Boltzmann population of the chosen conformer among the refined set. Above about 0.7 one conformer dominates and single-conformer DFT properties are representative. Below about 0.3 the molecule is genuinely flexible; say so in the paper, or re-run DFT on the second conformer for comparison.

In [ ]:
#@title Cell 6.2 — Run conformer search
run_stage("conformers", conformer_search)
_rows = []
for rec in MOLS:
    c = RES[rec["name"]].get("conformers", {})
    if c.get("done"):
        good = [x for x in c["conformers"] if x.get("rel_kcal") is not None]
        _rows.append({"molecule": rec["name"], "method": c["method"], "refined": len(c["conformers"]),
                      "valid (bonding intact)": len(good),
                      "ΔE range (kcal/mol)": round(max((x["rel_kcal"] for x in good), default=0), 2),
                      "pop. of best": round(c["best_population"], 3) if c.get("best_population") is not None else None})
CONF_TABLE = pd.DataFrame(_rows)
save_table(CONF_TABLE, "03_conformer_search")

### 📘 For Your Read — Cell 6.3: Conformer energy figure

**How to read it.** Each dot is one xTB-refined conformer at its relative energy; dot size scales with Boltzmann population, and the filled dot is the conformer sent to DFT. A tight cluster near 0 kcal/mol means conformational ambiguity; a lone dot at 0 with others above 2 kcal/mol means a clear global minimum.

In [ ]:
#@title Cell 6.3 — Conformer energy figure
_ok = [r for r in MOLS if RES[r["name"]].get("conformers", {}).get("done")]
if _ok:
    fig, ax = plt.subplots(figsize=(max(4.5, 1.3 * len(_ok) + 2), 3.4))
    for i, rec in enumerate(_ok):
        cs = [c for c in RES[rec["name"]]["conformers"]["conformers"] if c.get("rel_kcal") is not None]
        if not cs:
            continue
        rel = np.array([c["rel_kcal"] for c in cs]); pop = np.array([c["boltzmann"] for c in cs])
        jitter = (np.arange(len(rel)) - (len(rel) - 1) / 2) * 0.06
        ax.scatter(i + jitter, rel, s=30 + 220 * pop, facecolor="white", edgecolor=mol_color(i), lw=1.6, zorder=3)
        b = int(np.argmin(rel))
        ax.scatter(i + jitter[b], rel[b], s=30 + 220 * pop[b], color=mol_color(i), zorder=4)
    ax.set_xticks(range(len(_ok))); ax.set_xticklabels([r["name"] for r in _ok], rotation=20, ha="right")
    ax.set_ylabel("Relative GFN2-xTB energy (kcal/mol)"); ax.grid(axis="x", visible=False)
    ax.set_title("Conformer energies after xTB refinement (filled = selected for DFT)", loc="left")
    fig.tight_layout()
    save_fig(fig, "03_conformer_energies",
             "Relative GFN2-xTB energies of refined conformers; marker area ∝ Boltzmann population, filled marker = conformer used for DFT.", 6)

---
## Part 7 — DFT Geometry Optimisation

### 📘 For Your Read — Cell 7.1: Optimise all molecules

**What it does.** Optimises each xTB structure at the optimisation level of theory (Part 2), checkpointing every cycle. This is usually the longest stage. On a GPU, small drug-like molecules typically take minutes; larger or flexible ones can take much longer, particularly on CPU.

**How to read the table.** *Converged* must be **True**. *RMSD xTB→DFT* below about 0.3 Å means xTB already gave a good geometry; a large RMSD means the molecule rearranged (e.g. an intramolecular H-bond formed or broke), which is worth inspecting in the 3D viewer later.

**If it does not converge.** Re-run the cell; it resumes from the last geometry with another `OPT_MAX_STEPS` cycles. Persistent oscillation usually means a very flat torsional surface; the final geometry is then still usable for properties but the frequency check (Part 8) becomes essential.

In [ ]:
#@title Cell 7.1 — DFT geometry optimisation
def dft_optimize(rec):
    start = np.array(RES[rec["name"]]["conformers"]["best_xyz"])
    o = optimize_geometry(rec["symbols"], start, rec["charge"], rec["mult"], CFG["opt_func"], CFG["opt_basis"],
                          "opt", ckpt(rec))
    write_xyz(os.path.join(STRUCT_DIR, f"{rec['name']}_dft_opt.xyz"), rec["symbols"], o["coords"],
              f"{rec['name']} {lot(CFG['opt_func'], CFG['opt_basis'])} E={o['energy']:.10f}")
    if not o["converged"]:
        print(f"      ⚠️ {rec['name']}: not converged within {OPT_MAX_STEPS} cycles — re-run to continue")
    return dict(converged=o["converged"], energy=o["energy"], coords=o["coords"].tolist(), cycles=len(o["trace"]),
                trace=o["trace"], rmsd_xtb=kabsch_rmsd(start, o["coords"]))

run_stage("opt", dft_optimize, requires=("conformers",))
for rec in MOLS:        # an unconverged optimisation is kept but will be continued on the next run
    o = RES[rec["name"]].get("opt", {})
    if o.get("done") and not o.get("converged"):
        o["done"] = False
save_state()
_rows = []
for rec in MOLS:
    o = RES[rec["name"]].get("opt", {})
    if "energy" in o:
        _rows.append({"molecule": rec["name"], "E (Eh)": round(o["energy"], 8), "cycles": o["cycles"],
                      "converged": o["converged"], "RMSD xTB→DFT (Å)": round(o["rmsd_xtb"], 3),
                      "time": fmt_time(o.get("seconds", 0))})
OPT_TABLE = pd.DataFrame(_rows)
save_table(OPT_TABLE, "04_geometry_optimisation")

### 📘 For Your Read — Cell 7.2: Optimisation convergence figure

**How to read it.** Left: energy above the final energy (kcal/mol, log scale) per cycle. A smooth, monotonic descent is ideal. Right: RMS gradient per cycle with the convergence threshold dashed. Long flat tails mean soft torsional modes, which are normal for flexible molecules.

In [ ]:
#@title Cell 7.2 — Convergence traces
_ok = [(i, r) for i, r in enumerate(MOLS) if RES[r["name"]].get("opt", {}).get("trace")]
if _ok:
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.4))
    for i, rec in _ok:
        tr = RES[rec["name"]]["opt"]["trace"]
        e = np.array([t["energy"] for t in tr]); cyc = np.arange(1, len(e) + 1)
        de = (e - e.min()) * HARTREE2KCAL + 1e-4
        a1.plot(cyc, de, color=mol_color(i), marker="o", ms=3, lw=1.6, label=rec["name"])
        a2.plot(cyc, [t["grad_rms"] for t in tr], color=mol_color(i), marker="o", ms=3, lw=1.6, label=rec["name"])
    a1.set_yscale("log"); a1.set_xlabel("Optimisation cycle"); a1.set_ylabel("E − E$_{final}$ (kcal/mol)")
    a1.set_title("Energy convergence", loc="left")
    a2.axhline(3e-4, color=INK2, ls="--", lw=1); a2.text(1, 3.3e-4, "RMS-force threshold", fontsize=7.5, color=INK2)
    a2.set_yscale("log"); a2.set_xlabel("Optimisation cycle"); a2.set_ylabel("RMS gradient (Eh/bohr)")
    a2.set_title("Gradient convergence", loc="left")
    a2.legend(loc="upper right", fontsize=7.5)
    fig.tight_layout()
    save_fig(fig, "04_optimisation_convergence",
             f"Geometry optimisation convergence at {lot(CFG['opt_func'], CFG['opt_basis'])}.", 7)

---
## Part 8 — Vibrational Frequencies, Thermochemistry and IR Spectra

### 📘 For Your Read — Cell 8.1: Frequency functions

**What it does.**
- Computes the **analytic Hessian** at the optimised geometry, at exactly the optimisation level. Frequencies are only valid at a stationary point of the same method.
- **Harmonic analysis** with projection of translations and rotations, then **RRHO thermochemistry** (zero-point energy, enthalpy, entropy, Gibbs energy at `TEMPERATURE_K` and 1 atm).
- **IR intensities** from analytic atomic polar tensors (dipole derivatives via the coupled-perturbed Kohn–Sham response). On GPU the GPU4PySCF routine is used; on CPU an equivalent routine, checked against finite differences during development, is used. Intensities are computed for closed-shell gas-phase calculations; with PCM, frequencies are still reported but intensities are omitted, because the solvent response of the dipole derivatives is not implemented in these libraries.
- **Imaginary-frequency check.** A genuine minimum has none. With `AUTO_FIX_IMAGINARY`, a structure with an imaginary mode below −30 cm⁻¹ is displaced along that mode and re-optimised once. Modes between 0 and −30i cm⁻¹ are typically numerical noise from the integration grid in very flat torsions and are reported but not acted on.

In [ ]:
#@title Cell 8.1 — Frequency, thermochemistry & IR functions
from scipy.constants import physical_constants as _pc

def _ir_intensity_cpu(mf, norm_mode):
    """Analytic IR intensities (km/mol) for closed-shell CPU KS objects from atomic polar tensors."""
    mol = mf.mol; natm, nao = mol.natm, mol.nao
    hobj = mf.Hessian()
    mo_c, mo_o, mo_e = mf.mo_coeff, mf.mo_occ, mf.mo_energy
    mocc = mo_c[:, mo_o > 0]
    dm0 = mf.make_rdm1()
    atmlst = range(natm)
    h1ao = hobj.make_h1(mo_c, mo_o, None, atmlst)
    mo1, _ = hobj.solve_mo1(mo_e, mo_c, mo_o, h1ao, None, atmlst, hobj.max_memory, lib.logger.new_logger(hobj, 0))
    with mol.with_common_orig((0, 0, 0)):
        hmu = mol.intor("int1e_r")
        hmu11 = -mol.intor("int1e_irp").reshape(3, 3, nao, nao)
    aos = mol.aoslice_by_atom()
    apt = np.empty((3, 3, natm))
    for ia in atmlst:
        p0, p1 = aos[ia][2:]
        h11 = np.zeros((3, 3, nao, nao))
        h11[:, :, :, p0:p1] += hmu11[:, :, :, p0:p1]
        h11[:, :, p0:p1] += hmu11[:, :, :, p0:p1].transpose(0, 1, 3, 2)
        dm1 = np.einsum("ypi,vi->ypv", np.asarray(mo1[ia]), mocc)
        apt[:, :, ia] = -np.einsum("xuv,yuv->xy", hmu, dm1) * 4
        apt[:, :, ia] -= np.einsum("xyuv,vu->xy", h11, dm0)
        apt[:, :, ia] += mol.atom_charge(ia) * np.eye(3)
    unit = (_pc["fine-structure constant"][0] ** 2 * (1e-3 / _pc["atomic mass constant"][0]) * _pc["electron mass"][0]
            * _pc["Avogadro constant"][0] * np.pi * _pc["Bohr radius"][0] / 3)
    I = np.einsum("xym,nmy->xn", apt, norm_mode)
    return (I ** 2).sum(0) * unit

def _real_freqs(fw):
    fw = np.asarray(fw)
    if np.iscomplexobj(fw):
        return np.where(np.abs(fw.imag) > 1e-8, -np.abs(fw.imag), fw.real)
    return fw.astype(float)

def frequency_analysis(mf):
    mol = mf.mol
    closed = mol.spin == 0
    gas = SOLVENT == "none"
    inten = None
    if is_gpu_obj(mf) and closed and gas:
        from gpu4pyscf.properties import ir as gir
        hobj = mf.Hessian()
        f_ir, i_ir = gir.eval_ir_freq_intensity(mf, hobj)
        H = to_np(hobj.de)
        f_ir, i_ir = to_np(f_ir), to_np(i_ir)
    else:
        H = to_np(mf.Hessian().kernel())
    fa = thermo.harmonic_analysis(mol, H)
    freqs = _real_freqs(fa["freq_wavenumber"])
    if closed and gas:
        if is_gpu_obj(mf):
            if len(f_ir) == len(freqs) and np.allclose(np.sort(np.real(f_ir)), np.sort(freqs), atol=2.0):
                inten = np.asarray(i_ir)[np.argsort(np.real(f_ir))][np.argsort(np.argsort(freqs))]
        else:
            inten = _ir_intensity_cpu(mf, fa["norm_mode"])
    fs = CFG["freq_scale"]
    th = thermo.thermo(mf, np.asarray(fa["freq_au"]) * fs, TEMPERATURE_K, 101325)
    g = lambda k: float(np.real(th[k][0]))
    return dict(freqs=freqs, intens=None if inten is None else np.asarray(inten, float),
                norm_mode=np.real(fa["norm_mode"]),
                ZPE=g("ZPE"), E_tot=g("E_tot"), H_tot=g("H_tot"), G_tot=g("G_tot"),
                S_tot=g("S_tot"), Cv_tot=g("Cv_tot"), E_elec=float(mf.e_tot))

print("Frequency module ready — IR intensities:",
      "analytic (gas phase, closed shell)" if SOLVENT == "none" else "not available with PCM (frequencies only)")

### 📘 For Your Read — Cell 8.2: Run frequencies with the imaginary-mode check

**What you should see.** Per molecule: the number of imaginary frequencies (should be **0**), the lowest frequency, and thermochemistry. If a molecule had a significant imaginary mode and was repaired, the optimised geometry files are updated and later stages use the corrected structure.

**Reporting.** ZPE, H and G here are needed for reaction or binding free-energy differences; the absolute values are not meaningful alone. Report the temperature, the scale factor used, and the statement "all structures were confirmed as minima by the absence of imaginary frequencies".

In [ ]:
#@title Cell 8.2 — Run frequencies
def freq_stage(rec):
    coords = np.array(RES[rec["name"]]["opt"]["coords"])
    mf = mf_for(rec, coords, CFG["opt_func"], CFG["opt_basis"])
    fr = frequency_analysis(mf)
    fixed = False
    n_imag = int((fr["freqs"] < -30).sum())
    if n_imag and AUTO_FIX_IMAGINARY:
        k = int(np.argmin(fr["freqs"]))
        mode = fr["norm_mode"][k] / np.abs(fr["norm_mode"][k]).max() * 0.15      # 0.15 Å max displacement
        print(f"      imaginary mode {fr['freqs'][k]:.0f} cm⁻¹ → displacing and re-optimising")
        for f in glob.glob(os.path.join(ckpt(rec), "fix_*")):
            os.remove(f)
        o = optimize_geometry(rec["symbols"], coords + mode, rec["charge"], rec["mult"],
                              CFG["opt_func"], CFG["opt_basis"], "fix", ckpt(rec))
        coords = o["coords"]
        fr = frequency_analysis(o["mf"])
        RES[rec["name"]]["opt"].update(coords=coords.tolist(), energy=o["energy"])
        write_xyz(os.path.join(STRUCT_DIR, f"{rec['name']}_dft_opt.xyz"), rec["symbols"], coords,
                  f"{rec['name']} {lot(CFG['opt_func'], CFG['opt_basis'])} (imaginary mode removed)")
        fixed = True
    real = fr["freqs"]
    out = dict(freqs=real.tolist(), intens=None if fr["intens"] is None else fr["intens"].tolist(),
               n_imag=int((real < -30).sum()), n_small_imag=int(((real < 0) & (real >= -30)).sum()),
               lowest=float(real.min()), fixed_imaginary=fixed,
               **{k: fr[k] for k in ("ZPE", "E_tot", "H_tot", "G_tot", "S_tot", "Cv_tot", "E_elec")})
    return out

if RUN_FREQUENCIES:
    run_stage("freq", freq_stage, requires=("opt",))
else:
    print("Frequencies skipped (RUN_FREQUENCIES = False).")

### 📘 For Your Read — Cell 8.3: Thermochemistry table

**Columns.** *ZPE* zero-point vibrational energy; *H* and *G* total enthalpy and Gibbs free energy (electronic + thermal, Eh) at the stated temperature; *S* entropy; *Cv* heat capacity; *n imag* imaginary modes (< −30i cm⁻¹); *small imag* noise-level modes. Use differences in G between species, never absolute G, for any chemical conclusion.

In [ ]:
#@title Cell 8.3 — Thermochemistry table
_rows = []
for rec in MOLS:
    f = RES[rec["name"]].get("freq", {})
    if f.get("done"):
        _rows.append({"molecule": rec["name"], "E_elec (Eh)": round(f["E_elec"], 8),
                      "ZPE (kcal/mol)": round(f["ZPE"] * HARTREE2KCAL, 2), "H (Eh)": round(f["H_tot"], 8),
                      "G (Eh)": round(f["G_tot"], 8), "S (cal/mol·K)": round(f["S_tot"] * HARTREE2KCAL * 1000, 2),
                      "Cv (cal/mol·K)": round(f["Cv_tot"] * HARTREE2KCAL * 1000, 2), "n imag": f["n_imag"],
                      "small imag": f["n_small_imag"], "lowest ν (cm⁻¹)": round(f["lowest"], 1),
                      "repaired": f["fixed_imaginary"]})
THERMO_TABLE = pd.DataFrame(_rows)
if len(THERMO_TABLE):
    save_table(THERMO_TABLE, "05_thermochemistry")
    print(f"T = {TEMPERATURE_K} K, p = 1 atm, frequency scale factor = {CFG['freq_scale']}")
    if (THERMO_TABLE["n imag"] > 0).any():
        print("⚠️  Some structures still have imaginary modes — they are saddle points, not minima. Inspect them before reporting.")

### 📘 For Your Read — Cell 8.4: IR spectra

**How to read it.** One panel per molecule, wavenumber decreasing to the right as in experimental IR. Bands are Lorentzian-broadened (FWHM 20 cm⁻¹) from the scaled harmonic frequencies and analytic intensities; the five strongest bands are labelled. Compare positions with FT-IR data after scaling. Typical diagnostic regions: O–H/N–H 3200–3600, C–H 2850–3100, C=O 1650–1800, aromatic C=C 1450–1600 cm⁻¹.

With PCM (no intensities), the panel shows stick positions only.

In [ ]:
#@title Cell 8.4 — IR spectra
_ok = [(i, r) for i, r in enumerate(MOLS) if RES[r["name"]].get("freq", {}).get("done")]
if _ok:
    fig, axes = plt.subplots(len(_ok), 1, figsize=(8, 2.1 * len(_ok)), sharex=True, squeeze=False)
    x = np.linspace(400, 4000, 3601)
    ir_rows = []
    for ax, (i, rec) in zip(axes[:, 0], _ok):
        f = RES[rec["name"]]["freq"]
        nu = np.array(f["freqs"]) * CFG["freq_scale"]
        keep = nu > 0
        nu = nu[keep]
        if f["intens"] is not None:
            I = np.array(f["intens"])[keep]
            g = 10.0
            spec = (I[:, None] * (g / np.pi) / ((x[None, :] - nu[:, None]) ** 2 + g ** 2)).sum(0)
            ax.fill_between(x, spec / spec.max(), color=mol_color(i), alpha=0.18, lw=0)
            ax.plot(x, spec / spec.max(), color=mol_color(i), lw=1.4)
            top = np.argsort(I)[::-1][:5]
            for t in top:
                yv = spec[np.argmin(np.abs(x - nu[t]))] / spec.max()
                ax.annotate(f"{nu[t]:.0f}", (nu[t], yv), xytext=(0, 3), textcoords="offset points",
                            ha="center", fontsize=7, color=INK2)
            for n_, i_ in zip(nu, I):
                ir_rows.append({"molecule": rec["name"], "wavenumber (cm⁻¹)": round(n_, 1), "intensity (km/mol)": round(i_, 2)})
            ax.set_ylabel("Rel. absorbance")
        else:
            ax.vlines(nu, 0, 1, color=mol_color(i), lw=1)
            for n_ in nu:
                ir_rows.append({"molecule": rec["name"], "wavenumber (cm⁻¹)": round(n_, 1), "intensity (km/mol)": None})
            ax.set_ylabel("Positions")
        ax.set_ylim(0, 1.18); ax.set_xlim(4000, 400)
        ax.set_title(rec["name"], loc="left"); ax.grid(axis="y", visible=False)
    axes[-1, 0].set_xlabel("Wavenumber (cm⁻¹)" + ("" if CFG["freq_scale"] == 1.0 else f", scaled ×{CFG['freq_scale']}"))
    fig.tight_layout()
    save_fig(fig, "05_ir_spectra", f"Simulated IR spectra ({lot(CFG['opt_func'], CFG['opt_basis'])}, Lorentzian FWHM 20 cm⁻¹, scale {CFG['freq_scale']}).", 8)
    save_table(pd.DataFrame(ir_rows), "05_ir_bands", show=False)

---
## Part 9 — Single-Point Electronic Structure and Global Reactivity

### 📘 For Your Read — Cell 9.1: Single-point and N±1 calculations

**What it does.** At each optimised geometry it runs a single point at the higher single-point level (Part 2), giving more reliable orbital energies, dipoles and densities than the optimisation basis. If Fukui analysis is on, it also runs the **N+1 (anion)** and **N−1 (cation)** states vertically, at the same geometry and level, in their lowest spin state. All densities and orbitals are saved to `checkpoints/`, so later parts never repeat these calculations.

**Global conceptual-DFT descriptors** are computed two ways:
- **Koopmans (orbital) estimates:** I ≈ −E(HOMO), A ≈ −E(LUMO). These are the values most papers report.
- **ΔSCF (vertical) values:** I = E(N−1) − E(N), A = E(N) − E(N+1). These are physically better grounded, especially for A, which Koopmans estimates poorly.

From I and A: electronegativity χ = (I+A)/2, chemical potential μ = −χ, hardness η = (I−A)/2, softness S = 1/(2η), electrophilicity ω = μ²/(2η), maximum electron uptake ΔN_max = −μ/η, and the electro-accepting/donating powers ω⁺ = (I+3A)²/[16(I−A)] and ω⁻ = (3I+A)²/[16(I−A)].

**Convention note.** Some papers define η = I − A without the factor ½; state your convention. This notebook uses Parr–Pearson (η = (I−A)/2).

In [ ]:
#@title Cell 9.1 — Single-point & N±1 calculations
def _homo_lumo(mo_e, mo_occ):
    e, o = np.asarray(mo_e), np.asarray(mo_occ)
    if e.ndim == 1:
        return float(e[o > 0].max()), float(e[o == 0].min())
    occ = np.concatenate([e[s][o[s] > 0] for s in range(2)])
    vir = np.concatenate([e[s][o[s] == 0] for s in range(2)])
    return float(occ.max()), float(vir.min())

def _save_scf(path, mf):
    np.savez_compressed(path, mo_energy=to_np(mf.mo_energy), mo_coeff=to_np(mf.mo_coeff),
                        mo_occ=to_np(mf.mo_occ), dm=to_np(mf.make_rdm1()), e_tot=float(mf.e_tot),
                        mol=np.array(mf.mol.dumps()))

def load_scf(path):
    d = np.load(path, allow_pickle=False)
    mol = gto.loads(str(d["mol"]))
    mol.verbose, mol.max_memory = 0, MAX_MEMORY_MB
    return mol, {k: d[k] for k in d.files if k != "mol"}

def reactivity(I, A):
    chi = (I + A) / 2; eta = (I - A) / 2; mu = -chi
    return dict(I=I, A=A, chi=chi, mu=mu, eta=eta, S=1 / (2 * eta), omega=mu ** 2 / (2 * eta),
                dNmax=-mu / eta, omega_plus=(I + 3 * A) ** 2 / (16 * (I - A)), omega_minus=(3 * I + A) ** 2 / (16 * (I - A)))

def _lowest_mult(nelec):
    return 1 if nelec % 2 == 0 else 2

def sp_stage(rec):
    coords = np.array(RES[rec["name"]]["opt"]["coords"])
    mf = mf_for(rec, coords, CFG["sp_func"], CFG["sp_basis"])
    _save_scf(ckpt(rec, "sp_N.npz"), mf)
    ho, lu = _homo_lumo(to_np(mf.mo_energy), to_np(mf.mo_occ))
    dip = to_np(mf.dip_moment(unit="Debye", verbose=0))
    out = dict(E=float(mf.e_tot), HOMO=ho * HARTREE2EV, LUMO=lu * HARTREE2EV, gap=(lu - ho) * HARTREE2EV,
               dipole=float(np.linalg.norm(dip)), dipole_vec=dip.tolist(), nao=int(mf.mol.nao))
    out["koopmans"] = reactivity(-ho * HARTREE2EV, -lu * HARTREE2EV)
    if RUN_FUKUI:
        nel = mf.mol.nelectron
        mfa = mf_for(rec, coords, CFG["sp_func"], CFG["sp_basis"], charge=rec["charge"] - 1, mult=_lowest_mult(nel + 1))
        _save_scf(ckpt(rec, "sp_Nplus1.npz"), mfa)
        mfc = mf_for(rec, coords, CFG["sp_func"], CFG["sp_basis"], charge=rec["charge"] + 1, mult=_lowest_mult(nel - 1))
        _save_scf(ckpt(rec, "sp_Nminus1.npz"), mfc)
        I = (mfc.e_tot - mf.e_tot) * HARTREE2EV
        A = (mf.e_tot - mfa.e_tot) * HARTREE2EV
        out.update(E_Nplus1=float(mfa.e_tot), E_Nminus1=float(mfc.e_tot), deltaSCF=reactivity(I, A))
    return out

run_stage("sp", sp_stage, requires=("opt",))

### 📘 For Your Read — Cell 9.2: Global descriptor table

**How to read it.** Energies in eV unless stated.
- A **small HOMO–LUMO gap** and **low hardness η** indicate a soft, polarisable, more reactive molecule; a large gap indicates kinetic stability.
- **High ω** marks a good electrophile. A common empirical scale: ω > 1.5 eV strong, 0.8–1.5 moderate, < 0.8 marginal electrophile.
- **ω⁻ vs ω⁺** compares a molecule's tendency to donate versus accept charge, useful for ranking antioxidants (high ω⁻ = better electron donor).
- Compare Koopmans and ΔSCF columns: if they disagree strongly, prefer ΔSCF for I and A.

Two tables are written: `06_global_descriptors_koopmans.csv` and `06_global_descriptors_deltaSCF.csv`.

In [ ]:
#@title Cell 9.2 — Global descriptor tables
_lab = {"I": "I", "A": "A", "chi": "χ", "mu": "μ", "eta": "η", "S": "S (eV⁻¹)", "omega": "ω",
        "dNmax": "ΔN_max", "omega_plus": "ω⁺", "omega_minus": "ω⁻"}
def _desc_table(kind):
    rows = []
    for rec in MOLS:
        s = RES[rec["name"]].get("sp", {})
        if s.get("done") and kind in s:
            row = {"molecule": rec["name"], "E (Eh)": round(s["E"], 8), "HOMO": round(s["HOMO"], 3),
                   "LUMO": round(s["LUMO"], 3), "gap": round(s["gap"], 3), "dipole (D)": round(s["dipole"], 3)}
            row.update({_lab[k]: round(v, 3) for k, v in s[kind].items()})
            rows.append(row)
    return pd.DataFrame(rows)
DESC_K = _desc_table("koopmans")
if len(DESC_K):
    display(Markdown("**Koopmans (orbital-energy) descriptors** — eV"))
    save_table(DESC_K, "06_global_descriptors_koopmans")
DESC_D = _desc_table("deltaSCF")
if len(DESC_D):
    display(Markdown("**ΔSCF (vertical N±1) descriptors** — eV"))
    save_table(DESC_D, "06_global_descriptors_deltaSCF")

### 📘 For Your Read — Cell 9.3: Frontier-orbital energy diagram and density of states

**How to read it.**
- **Left: orbital energy diagram.** Levels from HOMO−4 to LUMO+4 for each molecule, with HOMO and LUMO emphasised and the gap annotated. Molecules with higher HOMOs are better electron donors (often relevant to antioxidant activity); lower LUMOs are better acceptors.
- **Right: density of states (DOS).** Orbital energies Gaussian-broadened (FWHM 0.3 eV). The occupied region is shaded; the dashed line marks the HOMO. It shows whether the frontier region has isolated orbitals or near-degenerate bands.

In [ ]:
#@title Cell 9.3 — Orbital energy diagram & DOS
_ok = [(i, r) for i, r in enumerate(MOLS) if RES[r["name"]].get("sp", {}).get("done")]
if _ok:
    fig = plt.figure(figsize=(11, 3.2 + 0.0 * len(_ok)))
    gs = fig.add_gridspec(1, 2, width_ratios=[1.1, 1])
    ax = fig.add_subplot(gs[0])
    for j, (i, rec) in enumerate(_ok):
        _, d = load_scf(ckpt(rec, "sp_N.npz"))
        e = d["mo_energy"] * HARTREE2EV; o = d["mo_occ"]
        if e.ndim == 2:
            e, o = e[0], o[0]
        nocc = int((o > 0).sum())
        for k in range(max(0, nocc - 5), min(len(e), nocc + 5)):
            hl = k in (nocc - 1, nocc)
            ax.hlines(e[k], j - 0.28, j + 0.28, color=mol_color(i) if hl else MUTED, lw=2.6 if hl else 1.0)
        ho, lu = e[nocc - 1], e[nocc]
        ax.annotate("", (j + 0.36, lu), (j + 0.36, ho), arrowprops=dict(arrowstyle="<->", color=INK2, lw=0.9))
        ax.text(j + 0.40, (ho + lu) / 2, f"{lu - ho:.2f} eV", va="center", fontsize=7.5, color=INK2)
    ax.set_xticks(range(len(_ok))); ax.set_xticklabels([r["name"] for _, r in _ok], rotation=20, ha="right")
    ax.set_xlim(-0.6, len(_ok) - 0.2); ax.set_ylabel("Orbital energy (eV)"); ax.grid(axis="x", visible=False)
    ax.set_title("Frontier orbital levels (HOMO−4 … LUMO+4)", loc="left")
    ax2 = fig.add_subplot(gs[1])
    grid_e = np.linspace(-14, 4, 1800); sig = 0.3 / 2.3548
    for i, rec in _ok:
        _, d = load_scf(ckpt(rec, "sp_N.npz"))
        e = np.atleast_2d(d["mo_energy"]) * HARTREE2EV; o = np.atleast_2d(d["mo_occ"])
        dos = np.exp(-((grid_e[None, :] - e.ravel()[:, None]) ** 2) / (2 * sig ** 2)).sum(0)
        ho = e[o > 0].max()
        ax2.plot(grid_e, dos, color=mol_color(i), lw=1.4, label=rec["name"])
        ax2.fill_between(grid_e, dos, where=grid_e <= ho, color=mol_color(i), alpha=0.12, lw=0)
        ax2.axvline(ho, color=mol_color(i), ls="--", lw=0.9)
    ax2.set_xlabel("Energy (eV)"); ax2.set_ylabel("DOS (arb.)"); ax2.set_title("Density of states (FWHM 0.3 eV)", loc="left")
    ax2.legend(fontsize=7.5); ax2.grid(axis="y", visible=False)
    fig.tight_layout()
    save_fig(fig, "06_orbital_levels_dos", f"Frontier orbital energies and density of states at {lot(CFG['sp_func'], CFG['sp_basis'])}.", 9)

---
## Part 10 — Atomic Charges and Local Reactivity (Fukui Functions)

### 📘 For Your Read — Cell 10.1: Hirshfeld charges and condensed Fukui functions

**What it does.**
- **Hirshfeld charges** partition the molecular density using spherically averaged free-atom densities, computed here with the same basis and core potentials as the molecule, and integrated on a DFT grid. Unlike Mulliken charges they barely depend on the basis set, which is why they are the standard choice for condensed Fukui functions. Mulliken charges are reported alongside for comparison with older literature.
- **Condensed Fukui functions** (finite-difference, frozen geometry):
  - f⁺ₖ = qₖ(N) − qₖ(N+1): susceptibility to **nucleophilic attack** (where an added electron goes).
  - f⁻ₖ = qₖ(N−1) − qₖ(N): susceptibility to **electrophilic attack** (where an electron is removed).
  - f⁰ₖ = (f⁺ₖ + f⁻ₖ)/2: radical attack.
  - **Dual descriptor** Δfₖ = f⁺ₖ − f⁻ₖ: Δf > 0 marks electrophilic sites (nucleophile attacks here), Δf < 0 marks nucleophilic sites (electrophile attacks here).
  - Local electrophilicity ωₖ⁺ = ω·f⁺ₖ and local softness sₖ± = S·fₖ± (ΔSCF-based ω, S).

**Sanity check printed per molecule.** Each Fukui function must sum to 1.00 over all atoms; the cell reports the sums.

In [ ]:
#@title Cell 10.1 — Hirshfeld charges & Fukui functions
from pyscf.scf import atom_hf
from pyscf.dft import numint
from scipy.interpolate import interp1d
_SPLINES = {}

def _free_atom_spline(mol, ia):
    lab = mol.atom_symbol(ia)
    key = (lab, CFG["sp_basis"])
    if key in _SPLINES:
        return _SPLINES[key]
    atm = gto.Mole()
    atm.atom = [[lab, (0.0, 0.0, 0.0)]]
    atm.basis = {lab: mol._basis[lab]}
    ncore = 0
    if mol._ecp and lab in mol._ecp:
        atm.ecp = {lab: mol._ecp[lab]}
        ncore = mol._ecp[lab][0]
    atm.spin = (gto.charge(lab) - ncore) % 2
    atm.verbose = 0
    atm.build()
    res = atom_hf.get_atm_nrhf(atm)
    _, _, c, occ = res[lab] if lab in res else list(res.values())[0]
    dmA = (c * occ) @ c.T
    r = np.concatenate([[0.0], np.logspace(-4, np.log10(25.0), 500)])
    pts = np.zeros((r.size, 3)); pts[:, 2] = r
    rho = np.maximum(numint.eval_rho(atm, numint.eval_ao(atm, pts), dmA), 1e-30)
    sp = interp1d(r, np.log(rho), bounds_error=False, fill_value=(np.log(rho[0]), -69.0))
    _SPLINES[key] = sp
    return sp

_GRID_CACHE = {}
def hirshfeld_charges(mol, dm):
    dm = np.asarray(dm)
    if dm.ndim == 3:
        dm = dm[0] + dm[1]
    key = mol.dumps()
    if key not in _GRID_CACHE:
        g = dft.gen_grid.Grids(mol); g.level = 3; g.build()
        pro = np.array([np.exp(_free_atom_spline(mol, i)(np.linalg.norm(g.coords - mol.atom_coord(i), axis=1)))
                        for i in range(mol.natm)])
        wts = pro / np.maximum(pro.sum(0), 1e-300)
        _GRID_CACHE.clear(); _GRID_CACHE[key] = (g, wts)
    g, wts = _GRID_CACHE[key]
    rho = np.empty(len(g.weights))
    for p0 in range(0, len(rho), 20000):
        ao = numint.eval_ao(mol, g.coords[p0:p0 + 20000])
        rho[p0:p0 + 20000] = numint.eval_rho(mol, ao, dm)
    return mol.atom_charges() - (wts * rho * g.weights).sum(1)

def mulliken_charges(mol, dm):
    dm = np.asarray(dm)
    if dm.ndim == 3:
        dm = dm[0] + dm[1]
    s = mol.intor_symmetric("int1e_ovlp")
    pop = np.einsum("ij,ji->i", dm, s)
    q = mol.atom_charges().astype(float).copy()
    for i, (b0, b1) in enumerate(mol.aoslice_by_atom()[:, 2:]):
        q[i] -= pop[b0:b1].sum()
    return q

def fukui_stage(rec):
    mol, dN = load_scf(ckpt(rec, "sp_N.npz"))
    q_N = hirshfeld_charges(mol, dN["dm"])
    q_mull = mulliken_charges(mol, dN["dm"])
    molp, dP = load_scf(ckpt(rec, "sp_Nplus1.npz"))
    molm, dM = load_scf(ckpt(rec, "sp_Nminus1.npz"))
    q_P = hirshfeld_charges(mol, dP["dm"])          # same geometry & basis → same grid and weights
    q_M = hirshfeld_charges(mol, dM["dm"])
    fp, fm = q_N - q_P, q_M - q_N
    s = RES[rec["name"]]["sp"]["deltaSCF"]
    tab = pd.DataFrame({"atom": [f"{a}{i+1}" for i, a in enumerate(rec["symbols"])], "index": np.arange(mol.natm),
                        "element": rec["symbols"], "q_Hirshfeld": q_N, "q_Mulliken": q_mull,
                        "f_plus": fp, "f_minus": fm, "f_zero": (fp + fm) / 2, "dual_df": fp - fm,
                        "omega_k_plus": s["omega"] * fp, "s_k_plus": s["S"] * fp, "s_k_minus": s["S"] * fm})
    tab.round(5).to_csv(os.path.join(TAB_DIR, f"07_fukui_{rec['name']}.csv"), index=False)
    print(f"      Σq={q_N.sum():+.4f} (charge {rec['charge']}), Σf⁺={fp.sum():.3f}, Σf⁻={fm.sum():.3f}")
    heavy = tab[tab.element != "H"]
    return dict(table=tab.round(6).to_dict(orient="list"),
                top_f_plus=heavy.nlargest(3, "f_plus")["atom"].tolist(),
                top_f_minus=heavy.nlargest(3, "f_minus")["atom"].tolist(),
                top_dual_pos=heavy.nlargest(3, "dual_df")["atom"].tolist(),
                top_dual_neg=heavy.nsmallest(3, "dual_df")["atom"].tolist())

if RUN_FUKUI:
    run_stage("fukui", fukui_stage, requires=("sp",))
else:
    print("Fukui analysis skipped (RUN_FUKUI = False).")

### 📘 For Your Read — Cell 10.2: Reactive-site maps and summary

**How to read it.** Left of each pair: the 2D structure with every heavy atom coloured by the dual descriptor, with hydrogen contributions summed onto their parent atom. **Blue (Δf > 0)** marks electrophilic sites where nucleophiles attack, e.g. carbonyl carbons; **red (Δf < 0)** marks nucleophilic sites where electrophiles attack, e.g. activated ring carbons, heteroatom lone pairs. Atom numbers match the CSV tables. Right: f⁺ and f⁻ per heavy atom.

**Interpretation in drug design.** High-f⁻ heteroatoms often coincide with metabolic oxidation sites and H-bond acceptors; high-f⁺ carbons flag Michael-acceptor or acylation reactivity that may raise covalent-binding or toxicity concerns. Treat these as hypotheses to check against docking poses and ADMET predictions.

In [ ]:
#@title Cell 10.2 — Reactive-site maps
def _heavy_condensed(rec, col):
    t = pd.DataFrame(RES[rec["name"]]["fukui"]["table"])
    m = rec["rdmol"]
    vals = {}
    for a in m.GetAtoms():
        if a.GetAtomicNum() == 1:
            continue
        v = t.loc[a.GetIdx(), col]
        v += sum(t.loc[n.GetIdx(), col] for n in a.GetNeighbors() if n.GetAtomicNum() == 1)
        vals[a.GetIdx()] = float(v)
    return vals

def _dual_map_png(rec, path):
    vals = _heavy_condensed(rec, "dual_df")
    m = Chem.RWMol(rec["rdmol"])
    for a in m.GetAtoms():
        a.SetIntProp("orig", a.GetIdx())
    m2 = Chem.RemoveHs(m)
    AllChem.Compute2DCoords(m2)
    vmax = max(1e-6, max(abs(v) for v in vals.values()))
    norm = TwoSlopeNorm(0, -vmax, vmax)
    cols, rad, idx = {}, {}, []
    for a in m2.GetAtoms():
        o = a.GetIntProp("orig")
        a.SetProp("atomNote", f"{a.GetSymbol()}{o+1}")
        cols[a.GetIdx()] = tuple(DIVERGING(norm(vals[o]))[:3])
        rad[a.GetIdx()] = 0.25 + 0.30 * abs(vals[o]) / vmax
        idx.append(a.GetIdx())
    d = rdMolDraw2D.MolDraw2DCairo(700, 560)
    op = d.drawOptions(); op.annotationFontScale = 0.55; op.padding = 0.08
    d.DrawMolecule(m2, highlightAtoms=idx, highlightAtomColors=cols, highlightAtomRadii=rad, highlightBonds=[])
    d.FinishDrawing()
    with open(path, "wb") as fh:
        fh.write(d.GetDrawingText())
    return vmax

_ok = [(i, r) for i, r in enumerate(MOLS) if RES[r["name"]].get("fukui", {}).get("done")]
for i, rec in _ok:
    png = os.path.join(FIG_DIR, f"_tmp_dual_{rec['name']}.png")
    vmax = _dual_map_png(rec, png)
    fig = plt.figure(figsize=(10.5, 3.9))
    gs = fig.add_gridspec(1, 2, width_ratios=[1, 1.35])
    a0 = fig.add_subplot(gs[0]); a0.imshow(plt.imread(png)); a0.axis("off")
    a0.set_title(f"{rec['name']} — dual descriptor Δf (blue +, red −)", loc="left")
    sm = plt.cm.ScalarMappable(norm=TwoSlopeNorm(0, -vmax, vmax), cmap=DIVERGING)
    cb = fig.colorbar(sm, ax=a0, fraction=0.04, pad=0.01); cb.set_label("Δf (H-condensed)", fontsize=8)
    fp, fm = _heavy_condensed(rec, "f_plus"), _heavy_condensed(rec, "f_minus")
    keys = list(fp.keys())
    labels = [f"{rec['symbols'][k]}{k+1}" for k in keys]
    x = np.arange(len(keys)); w = 0.38
    a1 = fig.add_subplot(gs[1])
    a1.bar(x - w / 2 - 0.01, [fp[k] for k in keys], w, color=PAL[0], label="f⁺ (nucleophilic attack)")
    a1.bar(x + w / 2 + 0.01, [fm[k] for k in keys], w, color=PAL[1], label="f⁻ (electrophilic attack)")
    a1.set_xticks(x); a1.set_xticklabels(labels, rotation=90, fontsize=7)
    a1.set_ylabel("Condensed Fukui (H-summed)"); a1.legend(fontsize=7.5, loc="lower left", bbox_to_anchor=(0, 1.0), ncol=2)
    a1.grid(axis="x", visible=False); a1.set_title("Hirshfeld-condensed Fukui functions", loc="left", pad=22)
    fig.tight_layout()
    save_fig(fig, f"07_fukui_{rec['name']}",
             f"{rec['name']}: dual-descriptor map and condensed Fukui functions (Hirshfeld, {lot(CFG['sp_func'], CFG['sp_basis'])}).", 10, vector=False)
    os.remove(png)
if _ok:
    FUKUI_SUMMARY = pd.DataFrame([{"molecule": r["name"],
        "top f⁺ (nucleophile attacks)": ", ".join(RES[r["name"]]["fukui"]["top_f_plus"]),
        "top f⁻ (electrophile attacks)": ", ".join(RES[r["name"]]["fukui"]["top_f_minus"]),
        "most electrophilic (Δf>0)": ", ".join(RES[r["name"]]["fukui"]["top_dual_pos"]),
        "most nucleophilic (Δf<0)": ", ".join(RES[r["name"]]["fukui"]["top_dual_neg"])} for _, r in _ok])
    save_table(FUKUI_SUMMARY, "07_fukui_summary")

---
## Part 11 — Frontier Orbitals and Molecular Electrostatic Potential Surfaces

### 📘 For Your Read — Cell 11.1: Cube generation and MEP surface descriptors

**What it does.**
1. Writes Gaussian-format **cube files** (density, HOMO, LUMO, and a coarser ESP grid for the 3D viewer) to `cubes/`. They open directly in VMD, Avogadro, Chimera, GaussView and Multiwfn.
2. Extracts the **0.001 au electron-density isosurface** (the Bader van der Waals surface used for MEP analysis) by marching cubes, and evaluates the electrostatic potential **exactly at each surface vertex**, not interpolated from a coarse grid.
3. Computes the **Murray–Politzer surface descriptors** used in QSAR and solubility/permeability models: V_S,min and V_S,max, mean positive and negative potentials (V̄⁺, V̄⁻), local polarity Π, variances σ²₊, σ²₋, σ²_tot, the balance parameter ν, the positive/negative surface areas, and the molecular volume enclosed by the surface.

`CUBE_POINTS` sets the grid resolution (64³ is a good balance; 96³+ for very smooth figures).

In [ ]:
#@title Cell 11.1 — Cubes, surfaces & MEP descriptors
from skimage.measure import marching_cubes
ISO_DENS, ISO_ORB, ESP_GRID = 0.001, 0.03, 44
_GPU_ESP_CHECKED = {"ok": None}

def esp_at_points(mol, dm, pts):
    """Exact electrostatic potential (au) at points (bohr): nuclei minus electron density."""
    pts = np.ascontiguousarray(pts, float)
    dm = np.asarray(dm)
    if dm.ndim == 3:
        dm = dm[0] + dm[1]
    Vn = np.zeros(len(pts))
    for z, r in zip(mol.atom_charges(), mol.atom_coords()):
        Vn += z / np.maximum(np.linalg.norm(pts - r, axis=1), 1e-8)
    def _cpu(P):
        out = np.empty(len(P)); blk = max(16, int(2.5e7 / mol.nao ** 2))
        for p0 in range(0, len(P), blk):
            fm = gto.fakemol_for_charges(P[p0:p0 + blk])
            out[p0:p0 + blk] = np.einsum("ijp,ij->p", df.incore.aux_e2(mol, fm), dm)
        return out
    Ve = None
    if USE_GPU and _GPU_ESP_CHECKED["ok"] is not False:
        try:
            import cupy
            from gpu4pyscf.gto.int3c1e import int1e_grids
            Ve = np.concatenate([to_np(int1e_grids(mol, cupy.asarray(pts[p0:p0 + 50000]), dm=cupy.asarray(dm))).ravel()
                                 for p0 in range(0, len(pts), 50000)])
            if _GPU_ESP_CHECKED["ok"] is None:
                _GPU_ESP_CHECKED["ok"] = bool(np.allclose(Ve[:12], _cpu(pts[:12]), atol=1e-6))
                if not _GPU_ESP_CHECKED["ok"]:
                    Ve = None
        except Exception:
            _GPU_ESP_CHECKED["ok"], Ve = False, None
    if Ve is None:
        Ve = _cpu(pts)
    return Vn - Ve

def _isosurface(cube_obj, field, level):
    if not (field.min() < level < field.max()):
        return None, None
    sp = np.diag(cube_obj.box) / (np.array([cube_obj.nx, cube_obj.ny, cube_obj.nz]) - 1)
    v, f, _, _ = marching_cubes(field, level=level, spacing=tuple(sp))
    return v + cube_obj.boxorig, f

def surface_descriptors(verts_ang, faces, V_au, mol, symbols):
    v0, v1, v2 = (verts_ang[faces[:, k]] for k in range(3))
    A = 0.5 * np.linalg.norm(np.cross(v1 - v0, v2 - v0), axis=1)
    Vt = V_au[faces].mean(1) * HARTREE2KCAL
    pos, neg = Vt > 0, Vt < 0
    At = A.sum(); Vbar = (A * Vt).sum() / At
    Vp = (A[pos] * Vt[pos]).sum() / A[pos].sum() if pos.any() else 0.0
    Vm = (A[neg] * Vt[neg]).sum() / A[neg].sum() if neg.any() else 0.0
    s2p = (A[pos] * (Vt[pos] - Vp) ** 2).sum() / A[pos].sum() if pos.any() else 0.0
    s2m = (A[neg] * (Vt[neg] - Vm) ** 2).sum() / A[neg].sum() if neg.any() else 0.0
    s2t = s2p + s2m
    xyz = mol.atom_coords(unit="Ang")
    near = lambda p: (lambda k: f"{symbols[k]}{k+1}")(int(np.argmin(np.linalg.norm(xyz - p, axis=1))))
    kmin, kmax = int(np.argmin(V_au)), int(np.argmax(V_au))
    return {"V_S,min (kcal/mol)": float(V_au[kmin] * HARTREE2KCAL), "V_min near": near(verts_ang[kmin]),
            "V_S,max (kcal/mol)": float(V_au[kmax] * HARTREE2KCAL), "V_max near": near(verts_ang[kmax]),
            "V̄_S (kcal/mol)": float(Vbar), "V̄⁺ (kcal/mol)": float(Vp), "V̄⁻ (kcal/mol)": float(Vm),
            "Π (kcal/mol)": float((A * np.abs(Vt - Vbar)).sum() / At),
            "σ²₊": float(s2p), "σ²₋": float(s2m), "σ²_tot": float(s2t), "ν": float(s2p * s2m / s2t ** 2) if s2t > 0 else 0.0,
            "area total (Å²)": float(At), "area + (Å²)": float(A[pos].sum()), "area − (Å²)": float(A[neg].sum())}

def surfaces_stage(rec):
    mol, d = load_scf(ckpt(rec, "sp_N.npz"))
    dm = d["dm"]; dm_t = dm[0] + dm[1] if dm.ndim == 3 else dm
    n, nm = CUBE_POINTS, rec["name"]
    cden = cubegen.Cube(mol, n, n, n, margin=5.0)
    rho = cubegen.density(mol, os.path.join(CUBE_DIR, f"{nm}_density.cube"), dm_t, nx=n, ny=n, nz=n, margin=5.0)
    e, o, c = d["mo_energy"], d["mo_occ"], d["mo_coeff"]
    if e.ndim == 2:
        e, o, c = e[0], o[0], c[0]
    nocc = int((o > 0).sum())
    homo = cubegen.orbital(mol, os.path.join(CUBE_DIR, f"{nm}_HOMO.cube"), c[:, nocc - 1], nx=n, ny=n, nz=n, margin=5.0)
    lumo = cubegen.orbital(mol, os.path.join(CUBE_DIR, f"{nm}_LUMO.cube"), c[:, nocc], nx=n, ny=n, nz=n, margin=5.0)
    # coarse density + ESP pair for the interactive viewer and for external programs
    cg = cubegen.Cube(mol, ESP_GRID, ESP_GRID, ESP_GRID, margin=5.0)
    cubegen.density(mol, os.path.join(CUBE_DIR, f"{nm}_density_coarse.cube"), dm_t, nx=ESP_GRID, ny=ESP_GRID, nz=ESP_GRID, margin=5.0)
    cg.write(esp_at_points(mol, dm_t, cg.get_coords()).reshape(ESP_GRID, ESP_GRID, ESP_GRID),
             os.path.join(CUBE_DIR, f"{nm}_ESP.cube"), comment="Molecular electrostatic potential (au)")
    # exact ESP on the 0.001 au isosurface
    verts, faces = _isosurface(cden, rho, ISO_DENS)
    V = esp_at_points(mol, dm_t, verts)
    vox = np.prod(np.diag(cden.box) / (np.array([n, n, n]) - 1)) * BOHR2ANG ** 3
    desc = surface_descriptors(verts * BOHR2ANG, faces, V, mol, rec["symbols"])
    desc["volume (Å³)"] = float((rho > ISO_DENS).sum() * vox)
    np.savez_compressed(ckpt(rec, "surfaces.npz"), verts=verts, faces=faces, V=V, homo=homo, lumo=lumo,
                        boxorig=cden.boxorig, box=cden.box, n=n, homo_e=e[nocc - 1], lumo_e=e[nocc])
    return dict(mep=desc, n_vertices=int(len(verts)), gpu_esp=_GPU_ESP_CHECKED["ok"])

if RUN_SURFACES:
    run_stage("surfaces", surfaces_stage, requires=("sp",))
    _rows = [{"molecule": r["name"], **{k: (round(v, 2) if isinstance(v, float) else v) for k, v in RES[r["name"]]["surfaces"]["mep"].items()}}
             for r in MOLS if RES[r["name"]].get("surfaces", {}).get("done")]
    MEP_TABLE = pd.DataFrame(_rows)
    if len(MEP_TABLE):
        save_table(MEP_TABLE, "08_mep_surface_descriptors")
else:
    MEP_TABLE = pd.DataFrame()
    print("Surfaces skipped (RUN_SURFACES = False).")

### 📘 For Your Read — Cell 11.2: Publication renders: MEP, HOMO and LUMO

**How to read it.** One row per molecule, oriented so its longest axis runs left–right and tilted slightly so that orbital lobes above and below the ring plane are both visible.
- **MEP** on the 0.001 au density surface. **Red** regions are electron-rich (negative potential: lone pairs, carbonyl O, π-faces) and are H-bond-acceptor and electrophile-attack regions. **Blue** regions are electron-poor (positive potential: acidic H, polarised C–H) and are H-bond-donor and nucleophile-attack regions. The colour scale is symmetric about zero and clipped at the 98th percentile so a single extreme point does not wash out the map. Labelled points mark V_S,min and V_S,max.
- **HOMO / LUMO** lobes at ±0.03 au (blue = positive phase, red = negative phase) over the molecular skeleton. HOMO localisation shows where the molecule donates electrons; LUMO localisation shows where it accepts them. This should agree with the f⁻ and f⁺ maps in Part 10.

Renders are drawn with Matplotlib for reproducibility. For glossy figures, open the cube files in VMD or Chimera with the same isovalues.

In [ ]:
#@title Cell 11.2 — MEP / HOMO / LUMO renders
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
_LIGHT = np.array([0.35, 0.45, 0.82]) / np.linalg.norm([0.35, 0.45, 0.82])

def _shade(rgb, tri):
    n = np.cross(tri[:, 1] - tri[:, 0], tri[:, 2] - tri[:, 0])
    n /= np.maximum(np.linalg.norm(n, axis=1, keepdims=True), 1e-12)
    k = 0.50 + 0.50 * np.abs(n @ _LIGHT)
    return np.clip(rgb * k[:, None], 0, 1)

def _frame(rec, coords_ang):
    ctr = coords_ang.mean(0); R = principal_frame(coords_ang)
    return lambda X: (np.asarray(X) - ctr) @ R

def _skeleton(ax, rec, P):
    m = rec["rdmol"]
    for b in m.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        ax.plot(*zip(P[i], P[j]), color="#6b6a66", lw=1.6, zorder=10)
    for k, s in enumerate(rec["symbols"]):
        ax.scatter(*P[k], s=18 if s == "H" else 46, color=ELEMENT_COLORS.get(s, "#9a9a9a"),
                   edgecolor="#3a3a3a", linewidth=0.4, zorder=11, depthshade=False)

def _finish(ax, pts):
    lo, hi = pts.min(0), pts.max(0); c = (lo + hi) / 2; h = (hi - lo).max() / 2 * 1.02
    ax.set_xlim(c[0] - h, c[0] + h); ax.set_ylim(c[1] - h, c[1] + h); ax.set_zlim(c[2] - h, c[2] + h)
    ax.set_box_aspect((1, 1, 1)); ax.set_proj_type("ortho"); ax.view_init(elev=62, azim=-90); ax.set_axis_off()

_ok = [(i, r) for i, r in enumerate(MOLS) if RES[r["name"]].get("surfaces", {}).get("done")]
for i, rec in _ok:
    S = np.load(ckpt(rec, "surfaces.npz"))
    mol, _ = load_scf(ckpt(rec, "sp_N.npz"))
    xyz = mol.atom_coords(unit="Ang"); T = _frame(rec, xyz); P = T(xyz)
    fig = plt.figure(figsize=(12, 4.3))
    # ---- MEP
    ax = fig.add_subplot(1, 3, 1, projection="3d"); ax.computed_zorder = False
    verts = T(S["verts"] * BOHR2ANG); faces = S["faces"]; V = S["V"] * HARTREE2KCAL
    vlim = float(np.percentile(np.abs(V), 98)); norm = TwoSlopeNorm(0, -vlim, vlim)
    tri = verts[faces]
    rgb = DIVERGING(norm(V[faces].mean(1)))[:, :3]
    ax.add_collection3d(Poly3DCollection(tri, facecolors=_shade(rgb, tri), edgecolor="none", linewidths=0))
    for k, lab in ((int(np.argmin(V)), "min"), (int(np.argmax(V)), "max")):
        ax.scatter(*verts[k], s=28, color="black", zorder=20, depthshade=False)
        ax.text(*verts[k], f" V$_{{S,{lab}}}$ {V[k]:.1f}", fontsize=7, zorder=21)
    _finish(ax, verts)
    ax.set_title("MEP on 0.001 au surface", fontsize=9)
    cb = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=DIVERGING), ax=ax, orientation="horizontal", fraction=0.05, pad=0.0, shrink=0.6, aspect=25)
    cb.set_ticks(np.linspace(-vlim, vlim, 5)); cb.set_ticklabels([f"{t:.0f}" for t in np.linspace(-vlim, vlim, 5)])
    cb.set_label("ESP (kcal/mol)", fontsize=8)
    # ---- HOMO / LUMO
    cobj = cubegen.Cube(mol, int(S["n"]), int(S["n"]), int(S["n"]), margin=5.0)
    for col, (key, title) in enumerate((("homo", "HOMO"), ("lumo", "LUMO")), start=2):
        ax = fig.add_subplot(1, 3, col, projection="3d"); ax.computed_zorder = False
        allp, tris, cols_ = [P], [], []
        for sign, color in ((+1, PAL[0]), (-1, PAL[7])):     # both phases in ONE collection → correct depth sorting
            v, f = _isosurface(cobj, sign * S[key], ISO_ORB)
            if v is None:
                continue
            vv = T(v * BOHR2ANG); tri = vv[f]; allp.append(vv)
            tris.append(tri); cols_.append(_shade(np.tile(np.array(to_rgb(color)), (len(f), 1)), tri))
        if tris:
            ax.add_collection3d(Poly3DCollection(np.concatenate(tris), facecolors=np.concatenate(cols_),
                                                 edgecolor="none", linewidths=0, alpha=0.80))
        _skeleton(ax, rec, P)
        _finish(ax, np.vstack(allp))
        ax.set_title(f"{title}  ({float(S[key + '_e']) * HARTREE2EV:.2f} eV)", fontsize=9)
    fig.suptitle(rec["name"], x=0.01, ha="left", fontsize=10, fontweight="bold")
    fig.subplots_adjust(left=0.0, right=0.98, wspace=0.02, top=0.88, bottom=0.02)
    save_fig(fig, f"08_surfaces_{rec['name']}",
             f"{rec['name']}: MEP mapped on the 0.001 au density isosurface and frontier orbitals (±{ISO_ORB} au) at {lot(CFG['sp_func'], CFG['sp_basis'])}.",
             11, vector=False)

### 📘 For Your Read — Cell 11.3: Interactive 3D viewer

**What it does.** Opens a rotatable WebGL view (py3Dmol) of one molecule with three linked panels: MEP-coloured surface, HOMO, LUMO. Rotating one rotates all three. Choose the molecule with `VIEW_MOLECULE` (blank = first) and re-run the cell to switch. Use it to check orbital localisation and electrostatic hot-spots from any angle before choosing an orientation for publication figures.

In [ ]:
#@title Cell 11.3 — Interactive 3D viewer
VIEW_MOLECULE = ""  #@param {type:"string"}
try:
    import py3Dmol
    _cands = [r for r in MOLS if RES[r["name"]].get("surfaces", {}).get("done")]
    _rec = next((r for r in _cands if r["name"] == VIEW_MOLECULE), _cands[0] if _cands else None)
    if _rec is None:
        print("No surfaces available yet — run Part 11.1 first.")
    else:
        nm = _rec["name"]
        xyz = xyz_block(_rec["symbols"], np.array(RES[nm]["opt"]["coords"]), nm)
        rd = lambda f: open(os.path.join(CUBE_DIR, f"{nm}_{f}.cube")).read()
        vlim = abs(RES[nm]["surfaces"]["mep"]["V_S,min (kcal/mol)"]) / HARTREE2KCAL
        v = py3Dmol.view(width=960, height=340, viewergrid=(1, 3), linked=True)
        for k in range(3):
            v.addModel(xyz, "xyz", viewer=(0, k))
            v.setStyle({"stick": {"radius": 0.12}, "sphere": {"scale": 0.2}}, viewer=(0, k))
        v.addVolumetricData(rd("density_coarse"), "cube", {"isoval": ISO_DENS, "opacity": 0.92, "voldata": rd("ESP"),
                            "volformat": "cube", "volscheme": {"gradient": "rwb", "min": -vlim, "max": vlim}}, viewer=(0, 0))
        for k, f in ((1, "HOMO"), (2, "LUMO")):
            cube = rd(f)
            v.addVolumetricData(cube, "cube", {"isoval": ISO_ORB, "color": PAL[0], "opacity": 0.85}, viewer=(0, k))
            v.addVolumetricData(cube, "cube", {"isoval": -ISO_ORB, "color": PAL[7], "opacity": 0.85}, viewer=(0, k))
        v.zoomTo(); v.show()
        print(f"{nm}: left MEP (red negative, blue positive) · middle HOMO · right LUMO")
except ImportError:
    print("py3Dmol not installed — interactive view unavailable.")

---
## Part 12 — TD-DFT Excited States and UV-Vis Spectra

### 📘 For Your Read — Cell 12.1: Run TD-DFT

**What it does.** Computes the lowest `TD_NSTATES` singlet excited states (or states of the same spin for open-shell molecules) at the optimised geometry with the TD-DFT functional, by default CAM-B3LYP, and the single-point basis. Full linear-response TD-DFT (RPA) is used unless `TD_USE_TDA` is ticked; the Tamm–Dancoff approximation is cheaper and more robust for triplet instabilities but slightly overestimates intensities.

For each state it records excitation energy (eV and nm), oscillator strength f, and the dominant orbital transitions (weight ≥ 10 %), labelled relative to the frontier orbitals, e.g. *H→L 92 %* or *H−1→L+2 15 %*.

**Expectations.** Typical accuracy for valence π→π* bands is about 0.2–0.4 eV. Range-separated functionals (CAM-B3LYP, ωB97X-V) are much better than B3LYP for charge-transfer states but tend to blue-shift local excitations slightly. With PCM, equilibrium linear-response solvation is used.

**On GPU.** The TD-DFT eigensolver in GPU4PySCF sizes its working space from PySCF's `max_memory`, which is set from the machine's system RAM. On high-RAM runtimes that would ask for tens of GB of GPU memory, because the subspace matrices grow with the square of the number of occupied–virtual orbital pairs. This cell therefore fixes the subspace at 30 vectors per requested state, which converges to the same excitation energies while keeping the workspace well under 1 GB for drug-sized molecules; the size is printed for each molecule. Cached GPU memory is also released before each molecule. If a molecule still fails, the cell prints the CUDA error, retries with half the subspace, and then either falls back to CPU (`TD_CPU_FALLBACK`, on by default) or skips the molecule.

**Cost note.** On CPU, many states for molecules with more than about 40 atoms can take a long time. Reduce `TD_NSTATES` if needed, since the visible region rarely needs more than 10 states for small chromophores.

In [ ]:
#@title Cell 12.1 — TD-DFT
def _orb_label(k, nocc):
    return ("H" if k == nocc - 1 else f"H−{nocc - 1 - k}") if k < nocc else ("L" if k == nocc else f"L+{k - nocc}")

def _contributions(xy, mo_occ, thresh=0.10):
    x = xy[0]
    out = []
    if isinstance(x, (tuple, list)):                     # unrestricted: (x_alpha, x_beta)
        for s, xs in zip("αβ", x):
            xs = to_np(xs); occ = np.asarray(mo_occ)[0 if s == "α" else 1]; nocc = int((occ > 0).sum())
            for (i, a), w in zip(np.argwhere(xs ** 2 >= thresh), xs[np.abs(xs) ** 2 >= thresh] ** 2):
                out.append((float(w), f"{_orb_label(i, nocc)}→{_orb_label(nocc + a, nocc)}({s})"))
    else:
        x = to_np(x); nocc = int((np.asarray(mo_occ) > 0).sum())
        w2 = 2 * x ** 2
        for i, a in np.argwhere(w2 >= thresh):
            out.append((float(w2[i, a]), f"{_orb_label(i, nocc)}→{_orb_label(nocc + a, nocc)}"))
    out.sort(reverse=True)
    return "; ".join(f"{lab} {100 * w:.0f}%" for w, lab in out[:4])

TD_SPACE_PER_STATE = 30     # Davidson subspace vectors kept per requested state (ample for convergence)

def _td_workspace(mf, nroots, space=None):
    """GPU4PySCF's TD eigensolver pre-allocates its subspace from `max_memory`, which PySCF takes from system RAM.
    On high-RAM runtimes that asks for tens of GB of GPU memory (the subspace matrices grow with (n_occ x n_vir)^2).
    Return a max_memory (MB) that instead gives a fixed subspace of `space` vectors, plus the size of that workspace."""
    occ = to_np(mf.mo_occ)
    occ = occ[None, :] if occ.ndim == 1 else occ
    A = int(sum((o > 0).sum() * (o == 0).sum() for o in occ))      # number of occupied-virtual pairs
    if space is None:
        space = TD_SPACE_PER_STATE * nroots
    space = int(min(A, max(4 * nroots + 10, space)))
    max_memory_mb = (space + nroots) * 64 * A / 1e6 + 1.0             # inverse of the solver's sizing rule
    workspace_gb = (32 * A * space + 128 * space ** 2) / 1e9           # 4 holders (A x space) + 4 matrices (2space)^2
    return max_memory_mb, workspace_gb, space, A

def _run_td(mf, space=None):
    td = mf.TDA() if TD_USE_TDA else mf.TDDFT()
    td.nstates = TD_NSTATES
    td.verbose = 0
    if is_gpu_obj(mf):
        td.max_memory, ws_gb, space, A = _td_workspace(mf, TD_NSTATES, space)
        print(f"      GPU eigensolver: {A} occ–vir pairs, subspace {space} vectors, workspace {ws_gb:.2f} GB")
    td.kernel()
    return td

def td_stage(rec):
    coords = np.array(RES[rec["name"]]["opt"]["coords"])
    td = mf = None
    if USE_GPU:
        space = None
        for attempt in (1, 2):           # attempt 2: half the subspace, after releasing cached GPU memory
            free_gpu()
            try:
                mf = mf_for(rec, coords, TD_FUNC, CFG["sp_basis"])
                td = _run_td(mf, space)
                engine = "GPU"
                break
            except Exception as e:
                print(f"      GPU TD-DFT attempt {attempt} failed — {type(e).__name__}: {str(e)[:160]}")
                if mf is not None:
                    space = max(4 * TD_NSTATES + 10, _td_workspace(mf, TD_NSTATES)[2] // 2)
                td = mf = None
                if not gpu_healthy():
                    raise RuntimeError("the CUDA context is no longer usable. Use Runtime ▸ Restart session and run all "
                                       "cells again: finished work is restored from checkpoints.")
        if td is None and not TD_CPU_FALLBACK:
            raise RuntimeError("GPU TD-DFT failed twice (see the messages above). Set TD_CPU_FALLBACK = True to run "
                               "this molecule on CPU, which is correct but slow.")
    if td is None:
        if USE_GPU:
            print(f"      Falling back to CPU. At {TD_FUNC}/{CFG['sp_basis']} this can take a long time; "
                  f"interrupt and set TD_CPU_FALLBACK = False to skip instead.")
        free_gpu()
        mf = mf_for(rec, coords, TD_FUNC, CFG["sp_basis"], gpu=False)
        td = _run_td(mf)
        engine = "CPU"
    e_ev = to_np(td.e) * HARTREE2EV
    f = to_np(td.oscillator_strength())
    occ = to_np(mf.mo_occ)
    states = [dict(state=k + 1, E_eV=float(e_ev[k]), nm=float(EV2NM / e_ev[k]), f=float(f[k]),
                   transitions=_contributions(td.xy[k], occ)) for k in range(len(e_ev))]
    conv = np.atleast_1d(to_np(getattr(td, "converged", np.ones(len(e_ev), bool))))
    if not conv.all():
        print(f"      ⚠️ {int((~conv.astype(bool)).sum())} TD state(s) not fully converged")
    return dict(states=states, functional=TD_FUNC, method="TDA" if TD_USE_TDA else "TDDFT", engine=engine)

TD_CPU_FALLBACK = True  #@param {type:"boolean"}
#@markdown *If GPU TD-DFT fails for a molecule, compute it on CPU (correct but slow). Untick to skip that molecule instead.*
if RUN_TDDFT:
    run_stage("tddft", td_stage, requires=("opt",))
else:
    print("TD-DFT skipped (RUN_TDDFT = False).")

### 📘 For Your Read — Cell 12.2: UV-Vis spectra and transition tables

**How to read it.**
- **Top: simulated absorption spectrum**; λmax is the strongest band maximum inside the window, and a curve that only rises toward the 180 nm edge (bands from higher states) is not counted as a band (molar absorptivity ε) from Gaussian broadening of each transition (σ = 0.4 eV, the GaussView default) using ε(ν̃) = 1.3062974×10⁸ Σᵢ (fᵢ/σ) exp[−((ν̃−ν̃ᵢ)/σ)²] with ν̃ and σ in cm⁻¹. λmax is labelled.
- **Bottom: stick spectrum** of oscillator strengths, to show which states build each band. Dark states (f ≈ 0) are invisible in the top panel but still listed in the table.
- The **transition table** gives the orbital character of each state. A band dominated by H→L confirms that the FMO pictures in Part 11 explain the absorption.

Experimental spectra in solution are usually red-shifted relative to gas-phase TD-DFT; use the matching solvent setting before comparing.

In [ ]:
#@title Cell 12.2 — UV-Vis spectra & transitions
_ok = [(i, r) for i, r in enumerate(MOLS) if RES[r["name"]].get("tddft", {}).get("done")]
if _ok:
    lam = np.linspace(180, 700, 2081); nu = 1e7 / lam
    sig = 0.4 * 8065.544
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(8, 5.2), sharex=True, gridspec_kw={"height_ratios": [2.2, 1]})
    td_rows, f_vis_max, y_top = [], 0.0, 0.0
    for i, rec in _ok:
        st = RES[rec["name"]]["tddft"]["states"]
        nu_i = np.array([1e7 / s["nm"] for s in st]); f_i = np.array([s["f"] for s in st])
        eps = (1.3062974e8 * f_i[:, None] / sig * np.exp(-((nu[None, :] - nu_i[:, None]) / sig) ** 2)).sum(0)
        a1.plot(lam, eps / 1e3, color=mol_color(i), lw=1.8, label=rec["name"])
        y_top = max(y_top, eps[lam >= 200].max() / 1e3)
        # λmax = most intense *band maximum* inside the window (a curve rising into the 180 nm edge is not a band)
        peaks = [k for k in range(1, len(eps) - 1) if eps[k] >= eps[k - 1] and eps[k] > eps[k + 1] and eps[k] > 1e-3 * eps.max()]
        if peaks:
            k = max(peaks, key=lambda q: eps[q])
            RES[rec["name"]]["tddft"]["lambda_max"] = float(lam[k]); RES[rec["name"]]["tddft"]["eps_max"] = float(eps[k])
            a1.annotate(f"{lam[k]:.0f} nm", (lam[k], eps[k] / 1e3), xytext=(6, 4 + 11 * (i % 3)), textcoords="offset points",
                        fontsize=7.5, color=mol_color(i), arrowprops=dict(arrowstyle="-", color=mol_color(i), lw=0.6))
        else:
            RES[rec["name"]]["tddft"]["lambda_max"] = None
        vis = [s for s in st if s["nm"] >= 180]
        f_vis_max = max([f_vis_max] + [s["f"] for s in vis]) if vis else f_vis_max
        a2.vlines([s["nm"] for s in vis], 0, [s["f"] for s in vis], color=mol_color(i), lw=1.8)
        for s in st:
            td_rows.append({"molecule": rec["name"], "state": s["state"], "E (eV)": round(s["E_eV"], 3),
                            "λ (nm)": round(s["nm"], 1), "f": round(s["f"], 4), "major transitions": s["transitions"]})
    save_state()
    a1.set_ylim(0, max(y_top, 1e-3) * 1.35)
    a1.set_ylabel("ε (10³ L mol⁻¹ cm⁻¹)"); a1.legend(fontsize=7.5); a1.grid(axis="x", visible=False)
    a1.set_title(f"Simulated UV-Vis — {TD_FUNC}/{CFG['sp_basis']} {'TDA' if TD_USE_TDA else 'TD-DFT'}, {SOLV_TXT}, σ = 0.4 eV", loc="left")
    a2.set_ylabel("Osc. strength f"); a2.set_xlabel("Wavelength (nm)"); a2.grid(axis="x", visible=False)
    a2.set_xlim(180, 700); a2.set_ylim(0, max(0.02, f_vis_max * 1.15))
    _short = [r["name"] for _, r in _ok if min(s["nm"] for s in RES[r["name"]]["tddft"]["states"]) > 180]
    if _short:
        print(f"ℹ️  For {', '.join(_short)} the {TD_NSTATES} computed states do not reach 180 nm; bands below the "
              f"highest state are missing from the plot. Raise TD_NSTATES to extend the spectrum.")
    fig.tight_layout()
    save_fig(fig, "09_uv_vis_spectra", f"Simulated UV-Vis absorption spectra and oscillator strengths ({TD_FUNC}/{CFG['sp_basis']}).", 12)
    TD_TABLE = pd.DataFrame(td_rows)
    save_table(TD_TABLE, "09_tddft_transitions")

---
## Part 13 — Comparative Analysis Across Molecules

### 📘 For Your Read — Cell 13.1: Master results table and automatic highlights

**What it does.** Merges every result into one table (`10_master_summary.csv`, one row per molecule): thermochemistry, FMO energies, Koopmans and ΔSCF descriptors, dipole, MEP descriptors and λmax. This is the table to paste into a manuscript, after choosing columns.

The highlights below it are computed, not interpreted: they name the extreme molecule for each key property so you can check them against your chemical intuition or biological data. Descriptor columns use ΔSCF values where available, otherwise Koopmans.

In [ ]:
#@title Cell 13.1 — Master table & highlights
_rows = []
for rec in MOLS:
    r = RES.get(rec["name"], {}); row = {"molecule": rec["name"], "SMILES": rec["smiles"],
        "formula": rdMolDescriptors.CalcMolFormula(rec["rdmol"]), "charge": rec["charge"], "mult": rec["mult"]}
    if r.get("opt", {}).get("done"):
        row["E_opt (Eh)"] = r["opt"]["energy"]
    if r.get("freq", {}).get("done"):
        f = r["freq"]; row.update({"ZPE (kcal/mol)": f["ZPE"] * HARTREE2KCAL, "G (Eh)": f["G_tot"], "n imag": f["n_imag"]})
    if r.get("sp", {}).get("done"):
        s = r["sp"]; d = s.get("deltaSCF", s["koopmans"])
        row.update({"E_SP (Eh)": s["E"], "HOMO (eV)": s["HOMO"], "LUMO (eV)": s["LUMO"], "gap (eV)": s["gap"],
                    "dipole (D)": s["dipole"], "I (eV)": d["I"], "A (eV)": d["A"], "χ (eV)": d["chi"],
                    "η (eV)": d["eta"], "S (eV⁻¹)": d["S"], "ω (eV)": d["omega"], "ω⁺ (eV)": d["omega_plus"],
                    "ω⁻ (eV)": d["omega_minus"], "descriptor source": "ΔSCF" if "deltaSCF" in s else "Koopmans"})
    if r.get("surfaces", {}).get("done"):
        m = r["surfaces"]["mep"]
        row.update({"V_S,min (kcal/mol)": m["V_S,min (kcal/mol)"], "V_S,max (kcal/mol)": m["V_S,max (kcal/mol)"],
                    "Π (kcal/mol)": m["Π (kcal/mol)"], "σ²_tot": m["σ²_tot"], "ν": m["ν"],
                    "area (Å²)": m["area total (Å²)"], "volume (Å³)": m["volume (Å³)"]})
    if r.get("tddft", {}).get("done"):
        st = r["tddft"]["states"]; b = max(st, key=lambda s: s["f"])
        row.update({"λmax (nm)": r["tddft"].get("lambda_max"), "brightest state (nm)": b["nm"], "f_max": b["f"]})
    _rows.append(row)
MASTER = pd.DataFrame(_rows)
_num = MASTER.select_dtypes("number").columns
MASTER[_num] = MASTER[_num].round(4)
save_table(MASTER, "10_master_summary")

HIGHLIGHTS = []
def _hl(col, fn, text):
    if col in MASTER and MASTER[col].notna().sum() >= 2:
        k = getattr(MASTER[col], fn)()
        HIGHLIGHTS.append(f"- {text}: **{MASTER.loc[k, 'molecule']}** ({col} = {MASTER.loc[k, col]:.3g})")
_hl("gap (eV)", "idxmin", "Smallest HOMO–LUMO gap (softest, most polarisable)")
_hl("gap (eV)", "idxmax", "Largest HOMO–LUMO gap (kinetically most stable)")
_hl("ω (eV)", "idxmax", "Strongest electrophile")
_hl("ω⁻ (eV)", "idxmax", "Strongest electron donor (highest ω⁻; antioxidant ranking)")
_hl("dipole (D)", "idxmax", "Most polar")
_hl("V_S,min (kcal/mol)", "idxmin", "Most negative surface potential (strongest H-bond-acceptor site)")
_hl("λmax (nm)", "idxmax", "Most red-shifted absorption")
if HIGHLIGHTS:
    display(Markdown("**Computed highlights**\n" + "\n".join(HIGHLIGHTS)))

### 📘 For Your Read — Cell 13.2: Comparative figures

**How to read them** (shown when at least two molecules finished):
- **FMO ladder:** each bar spans HOMO → LUMO for one molecule; the bar length is the gap. Higher bar tops mean easier electron acceptance; higher bottoms mean easier donation.
- **Descriptor heatmap:** each column standardised (z-score) across molecules so properties with different units are comparable; cells show the raw values. Red = below the set average, blue = above.
- **Gap vs electrophilicity:** separates soft electrophiles (small gap, high ω) from hard, unreactive molecules (large gap, low ω).

In [ ]:
#@title Cell 13.2 — Comparative figures
if "HOMO (eV)" in MASTER and MASTER["HOMO (eV)"].notna().sum() >= 2:
    M = MASTER.dropna(subset=["HOMO (eV)"]).reset_index(drop=True)
    idx = [next(i for i, r in enumerate(MOLS) if r["name"] == n) for n in M["molecule"]]
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), gridspec_kw={"width_ratios": [1, 1.6, 1]})
    ax = axes[0]
    for j, (k, row) in enumerate(M.iterrows()):
        ax.bar(j, row["LUMO (eV)"] - row["HOMO (eV)"], bottom=row["HOMO (eV)"], width=0.5, color=mol_color(idx[j]))
        ax.text(j, row["LUMO (eV)"] + 0.1, f"{row['gap (eV)']:.2f}", ha="center", fontsize=7.5, color=INK2)
    ax.set_xticks(range(len(M))); ax.set_xticklabels(M["molecule"], rotation=25, ha="right")
    ax.set_ylim(M["HOMO (eV)"].min() - 0.8, M["LUMO (eV)"].max() + 0.8)
    ax.set_ylabel("Energy (eV)"); ax.set_title("HOMO → LUMO span (gap in eV)", loc="left"); ax.grid(axis="x", visible=False)
    cols = [c for c in ["gap (eV)", "η (eV)", "ω (eV)", "ω⁺ (eV)", "ω⁻ (eV)", "dipole (D)", "V_S,min (kcal/mol)",
                        "V_S,max (kcal/mol)", "Π (kcal/mol)", "λmax (nm)"] if c in M and M[c].notna().all()]
    Z = (M[cols] - M[cols].mean()) / M[cols].std(ddof=0).replace(0, 1)
    ax = axes[1]
    im = ax.imshow(Z.values, cmap=DIVERGING, norm=TwoSlopeNorm(0, -2, 2), aspect="auto")
    for (r_, c_), v in np.ndenumerate(M[cols].values):
        ax.text(c_, r_, f"{v:.2f}" if abs(v) < 100 else f"{v:.0f}", ha="center", va="center", fontsize=7, color=INK)
    ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, rotation=35, ha="right", fontsize=7.5)
    ax.set_yticks(range(len(M))); ax.set_yticklabels(M["molecule"]); ax.grid(False)
    ax.set_title("Descriptor heatmap (colour = z-score, text = value)", loc="left")
    fig.colorbar(im, ax=ax, fraction=0.03, pad=0.01).set_label("z-score", fontsize=8)
    ax = axes[2]
    for j, (k, row) in enumerate(M.iterrows()):
        ax.scatter(row["gap (eV)"], row["ω (eV)"], s=60, color=mol_color(idx[j]), edgecolor="white", lw=1.5, zorder=3)
        ax.annotate(row["molecule"], (row["gap (eV)"], row["ω (eV)"]), xytext=(5, 4), textcoords="offset points", fontsize=7.5)
    ax.set_xlabel("HOMO–LUMO gap (eV)"); ax.set_ylabel("Electrophilicity ω (eV)")
    ax.set_title("Reactivity map", loc="left")
    fig.tight_layout()
    save_fig(fig, "10_comparative_overview", "Cross-molecule comparison: FMO span, standardised descriptor heatmap and gap–electrophilicity map.", 13)
else:
    print("Comparative figures need at least two molecules with completed single points.")

---
## Part 14 — Export and Reporting

### 📘 For Your Read — Cell 14.1: Structures and data exports

**What it writes.**
- `structures/<name>_dft_opt.xyz` and a combined **`DFT_optimised_structures.sdf`** with DFT properties as SD tags (HOMO, LUMO, gap, ω, dipole, G…). The SDF goes straight into docking (Vina/GNINA ligand prep), pharmacophore or QSAR tools, so the other BinFoLab notebooks can start from DFT-quality geometries.
- **Molden files** (`molden/`) with orbitals and energies, viewable in Avogadro, Jmol, Chemcraft or Multiwfn.
- **`DFTFlow_results.xlsx`**, one workbook with a sheet per analysis.

In [ ]:
#@title Cell 14.1 — Structures & data exports
_w = Chem.SDWriter(os.path.join(STRUCT_DIR, "DFT_optimised_structures.sdf"))
for rec in MOLS:
    r = RES.get(rec["name"], {})
    if not r.get("opt", {}).get("done"):
        continue
    m = Chem.Mol(rec["rdmol"]); m.RemoveAllConformers()
    conf = Chem.Conformer(m.GetNumAtoms())
    for k, c in enumerate(r["opt"]["coords"]):
        conf.SetAtomPosition(k, c)
    m.AddConformer(conf, assignId=True)
    m.SetProp("_Name", rec["name"]); m.SetProp("level_of_theory", LEVEL_STRING)
    row = MASTER[MASTER.molecule == rec["name"]].iloc[0]
    for col, val in row.items():
        if col not in ("molecule", "SMILES") and pd.notna(val):
            m.SetProp(col, str(val))
    _w.write(m)
    if os.path.exists(ckpt(rec, "sp_N.npz")):
        mol, d = load_scf(ckpt(rec, "sp_N.npz"))
        e, o, c = d["mo_energy"], d["mo_occ"], d["mo_coeff"]
        if e.ndim == 2:
            e, o, c = e[0], o[0], c[0]
        molden.from_mo(mol, os.path.join(MOLDEN_DIR, f"{rec['name']}.molden"), c, ene=e, occ=o)
_w.close()

XLSX = os.path.join(TAB_DIR, "DFTFlow_results.xlsx")
with pd.ExcelWriter(XLSX, engine="openpyxl") as xw:
    pd.DataFrame([{"setting": k, "value": str(v)} for k, v in {**CFG, "level_of_theory": LEVEL_STRING}.items()]).to_excel(xw, sheet_name="Settings", index=False)
    for nm_, obj in [("Summary", "MASTER"), ("Input", "INPUT_TABLE"), ("Validation", "VT"), ("Conformers", "CONF_TABLE"),
                     ("Optimisation", "OPT_TABLE"), ("Thermochemistry", "THERMO_TABLE"), ("Descriptors_Koopmans", "DESC_K"),
                     ("Descriptors_dSCF", "DESC_D"), ("MEP", "MEP_TABLE"), ("TDDFT", "TD_TABLE")]:
        t = globals().get(obj)
        if isinstance(t, pd.DataFrame) and len(t):
            t.to_excel(xw, sheet_name=nm_[:31], index=False)
    _fk = [pd.read_csv(p).assign(molecule=os.path.basename(p)[9:-4]) for p in sorted(glob.glob(os.path.join(TAB_DIR, "07_fukui_*.csv")))
           if "summary" not in p]
    if _fk:
        pd.concat(_fk).to_excel(xw, sheet_name="Fukui_per_atom", index=False)
print("Wrote: optimised SDF + XYZ, Molden files, Excel workbook →", OUT_DIR)

### 📘 For Your Read — Cell 14.2: Auto-generated methods section and references

**What it does.** Writes a **Computational Methods** paragraph that matches exactly what was run (functional, basis, dispersion, solvent, conformer protocol, convergence criteria, temperature, scale factor, TD-DFT settings, software versions), with numbered citations to the original method papers. It is saved as `report/methods.md` and included in the PDF.

**Before submission.** Read it once against your journal's style. It is accurate to the run, but you may wish to shorten it or move details to the SI.

In [ ]:
#@title Cell 14.2 — Methods text & references
REFS = {
 "rdkit": "RDKit: Open-source cheminformatics software. https://www.rdkit.org",
 "etkdg": "Riniker, S.; Landrum, G. A. Better informed distance geometry: using what we know to improve conformation generation. J. Chem. Inf. Model. 2015, 55, 2562–2574.",
 "etkdg3": "Wang, S.; Witek, J.; Landrum, G. A.; Riniker, S. Improving conformer generation for small rings and macrocycles based on distance geometry and experimental torsional-angle preferences. J. Chem. Inf. Model. 2020, 60, 2044–2058.",
 "mmff": "Halgren, T. A. Merck molecular force field. I. Basis, form, scope, parameterization, and performance of MMFF94. J. Comput. Chem. 1996, 17, 490–519.",
 "gfn2": "Bannwarth, C.; Ehlert, S.; Grimme, S. GFN2-xTB—An accurate and broadly parametrized self-consistent tight-binding quantum chemical method with multipole electrostatics and density-dependent dispersion contributions. J. Chem. Theory Comput. 2019, 15, 1652–1671.",
 "alpb": "Ehlert, S.; Stahn, M.; Spicher, S.; Grimme, S. Robust and efficient implicit solvation model for fast semiempirical methods. J. Chem. Theory Comput. 2021, 17, 4250–4261.",
 "ase": "Larsen, A. H. et al. The atomic simulation environment—a Python library for working with atoms. J. Phys.: Condens. Matter 2017, 29, 273002.",
 "pyscf": "Sun, Q. et al. Recent developments in the PySCF program package. J. Chem. Phys. 2020, 153, 024109.",
 "gpu4pyscf": "Li, R.; Sun, Q.; Zhang, X.; Chan, G. K.-L. Introducing GPU acceleration into the Python-based simulations of chemistry framework. J. Phys. Chem. A 2025, 129, 1459–1468.",
 "gpu4pyscf2": "Wu, X.; Sun, Q.; Pu, Z.; Zheng, T.; Ma, W.; Yan, W.; Yu, X.; Wu, Z.; Huo, M.; Li, X.; Ren, W.; Gong, S.; Zhang, Y.; Gao, W. Enhancing GPU-acceleration in the Python-based simulations of chemistry framework. arXiv:2404.09452, 2024.",
 "libxc": "Lehtola, S.; Steigemann, C.; Oliveira, M. J. T.; Marques, M. A. L. Recent developments in libxc—A comprehensive library of functionals for density functional theory. SoftwareX 2018, 7, 1–5.",
 "b3lyp": "Becke, A. D. Density-functional thermochemistry. III. The role of exact exchange. J. Chem. Phys. 1993, 98, 5648–5652; Lee, C.; Yang, W.; Parr, R. G. Phys. Rev. B 1988, 37, 785–789; Stephens, P. J.; Devlin, F. J.; Chabalowski, C. F.; Frisch, M. J. J. Phys. Chem. 1994, 98, 11623–11627.",
 "pbe0": "Adamo, C.; Barone, V. Toward reliable density functional methods without adjustable parameters: the PBE0 model. J. Chem. Phys. 1999, 110, 6158–6170.",
 "m062x": "Zhao, Y.; Truhlar, D. G. The M06 suite of density functionals for main group thermochemistry, thermochemical kinetics, noncovalent interactions, excited states, and transition elements. Theor. Chem. Acc. 2008, 120, 215–241.",
 "camb3lyp": "Yanai, T.; Tew, D. P.; Handy, N. C. A new hybrid exchange–correlation functional using the Coulomb-attenuating method (CAM-B3LYP). Chem. Phys. Lett. 2004, 393, 51–57.",
 "wb97xv": "Mardirossian, N.; Head-Gordon, M. ωB97X-V: a 10-parameter, range-separated hybrid, generalized gradient approximation density functional with nonlocal correlation, designed by a survival-of-the-fittest strategy. Phys. Chem. Chem. Phys. 2014, 16, 9904–9924.",
 "d3": "Grimme, S.; Antony, J.; Ehrlich, S.; Krieg, H. A consistent and accurate ab initio parametrization of density functional dispersion correction (DFT-D) for the 94 elements H–Pu. J. Chem. Phys. 2010, 132, 154104.",
 "d3bj": "Grimme, S.; Ehrlich, S.; Goerigk, L. Effect of the damping function in dispersion corrected density functional theory. J. Comput. Chem. 2011, 32, 1456–1465.",
 "d4": "Caldeweyher, E.; Ehlert, S.; Hansen, A.; Neugebauer, H.; Spicher, S.; Bannwarth, C.; Grimme, S. A generally applicable atomic-charge dependent London dispersion correction. J. Chem. Phys. 2019, 150, 154122.",
 "def2": "Weigend, F.; Ahlrichs, R. Balanced basis sets of split valence, triple zeta valence and quadruple zeta valence quality for H to Rn: design and assessment of accuracy. Phys. Chem. Chem. Phys. 2005, 7, 3297–3305.",
 "jkfit": "Weigend, F. Accurate Coulomb-fitting basis sets for H to Rn. Phys. Chem. Chem. Phys. 2006, 8, 1057–1065.",
 "pople": "Hehre, W. J.; Ditchfield, R.; Pople, J. A. Self-consistent molecular orbital methods. XII. Further extensions of Gaussian-type basis sets for use in molecular orbital studies of organic molecules. J. Chem. Phys. 1972, 56, 2257–2261; Krishnan, R.; Binkley, J. S.; Seeger, R.; Pople, J. A. J. Chem. Phys. 1980, 72, 650–654.",
 "iefpcm": "Cancès, E.; Mennucci, B.; Tomasi, J. A new integral equation formalism for the polarizable continuum model: theoretical background and applications to isotropic and anisotropic dielectrics. J. Chem. Phys. 1997, 107, 3032–3041.",
 "pcmrev": "Tomasi, J.; Mennucci, B.; Cammi, R. Quantum mechanical continuum solvation models. Chem. Rev. 2005, 105, 2999–3094.",
 "geometric": "Wang, L.-P.; Song, C. Geometry optimization made simple with translation and rotation coordinates. J. Chem. Phys. 2016, 144, 214108.",
 "mcquarrie": "McQuarrie, D. A. Statistical Mechanics; University Science Books: Sausalito, CA, 2000.",
 "cccbdb": "Johnson, R. D., III, Ed. NIST Computational Chemistry Comparison and Benchmark Database, NIST Standard Reference Database 101. https://cccbdb.nist.gov/",
 "koopmans": "Koopmans, T. Über die Zuordnung von Wellenfunktionen und Eigenwerten zu den einzelnen Elektronen eines Atoms. Physica 1934, 1, 104–113.",
 "parrpearson": "Parr, R. G.; Pearson, R. G. Absolute hardness: companion parameter to absolute electronegativity. J. Am. Chem. Soc. 1983, 105, 7512–7516.",
 "omega": "Parr, R. G.; Szentpály, L. v.; Liu, S. Electrophilicity index. J. Am. Chem. Soc. 1999, 121, 1922–1924.",
 "gazquez": "Gázquez, J. L.; Cedillo, A.; Vela, A. Electrodonating and electroaccepting powers. J. Phys. Chem. A 2007, 111, 1966–1970.",
 "fukui": "Parr, R. G.; Yang, W. Density functional approach to the frontier-electron theory of chemical reactivity. J. Am. Chem. Soc. 1984, 106, 4049–4050.",
 "condensed": "Yang, W.; Mortier, W. J. The use of global and local molecular parameters for the analysis of the gas-phase basicity of amines. J. Am. Chem. Soc. 1986, 108, 5708–5711.",
 "dual": "Morell, C.; Grand, A.; Toro-Labbé, A. New dual descriptor for chemical reactivity. J. Phys. Chem. A 2005, 109, 205–212.",
 "hirshfeld": "Hirshfeld, F. L. Bonded-atom fragments for describing molecular charge densities. Theor. Chim. Acta 1977, 44, 129–138.",
 "mulliken": "Mulliken, R. S. Electronic population analysis on LCAO–MO molecular wave functions. I. J. Chem. Phys. 1955, 23, 1833–1840.",
 "bader": "Bader, R. F. W.; Carroll, M. T.; Cheeseman, J. R.; Chang, C. Properties of atoms in molecules: atomic volumes. J. Am. Chem. Soc. 1987, 109, 7968–7979.",
 "mep": "Murray, J. S.; Politzer, P. The electrostatic potential: an overview. WIREs Comput. Mol. Sci. 2011, 1, 153–163.",
 "mpdesc": "Murray, J. S.; Lane, P.; Brinck, T.; Paulsen, K.; Grice, M. E.; Politzer, P. Relationships of critical constants and boiling points to computed molecular surface properties. J. Phys. Chem. 1993, 97, 9369–9373.",
 "marching": "Lewiner, T.; Lopes, H.; Vieira, A. W.; Tavares, G. Efficient implementation of Marching Cubes' cases with topological guarantees. J. Graphics Tools 2003, 8, 1–15.",
 "skimage": "van der Walt, S. et al. scikit-image: image processing in Python. PeerJ 2014, 2, e453.",
 "rungegross": "Runge, E.; Gross, E. K. U. Density-functional theory for time-dependent systems. Phys. Rev. Lett. 1984, 52, 997–1000.",
 "casida": "Casida, M. E. Time-dependent density functional response theory for molecules. In Recent Advances in Density Functional Methods, Part I; Chong, D. P., Ed.; World Scientific: Singapore, 1995; pp 155–192.",
 "tda": "Hirata, S.; Head-Gordon, M. Time-dependent density functional theory within the Tamm–Dancoff approximation. Chem. Phys. Lett. 1999, 314, 291–299.",
 "3dmol": "Rego, N.; Koes, D. 3Dmol.js: molecular visualization with WebGL. Bioinformatics 2015, 31, 1322–1324.",
}
_order = []
def cite(*keys):
    nums = []
    for k in keys:
        if k not in _order:
            _order.append(k)
        nums.append(_order.index(k) + 1)
    return "[" + ",".join(map(str, sorted(set(nums)))) + "]"

FUNC_REF = {"B3LYP": "b3lyp", "PBE0": "pbe0", "M06-2X": "m062x", "CAM-B3LYP": "camb3lyp", "wB97X-V": "wb97xv"}
DISP_REF = {"D3(BJ)": ("d3", "d3bj"), "D3(0)": ("d3",), "D4": ("d4",), "none": ()}
def basis_ref(b): return "def2" if b.startswith("def2") else "pople"
def _func_txt(func, basis):
    d = disp_label(func)
    t = f"{lot(func, basis)} level ({func} functional"
    t += " including VV10 non-local correlation" if func == "wB97X-V" else ""
    t += f" {cite(FUNC_REF[func])}"
    if d != "none":
        t += f", Grimme's {d} dispersion correction {cite(*DISP_REF[d])}"
    return t + f", {basis} basis set {cite(basis_ref(basis))})"

METHODS = ["### Computational methods"]
_conf = (f"Starting structures were generated from SMILES with RDKit {cite('rdkit')} using the ETKDGv3 distance-geometry "
         f"algorithm {cite('etkdg', 'etkdg3')} ({N_CONFORMERS} conformers, 0.3 Å RMSD pruning), relaxed with the MMFF94 force field "
         f"{cite('mmff')}, and the {XTB_REFINE_TOP} lowest conformers were re-optimised at the GFN2-xTB level {cite('gfn2')} with "
         f"tblite through ASE {cite('ase')}")
if SOLVENT != "none":
    _conf += f" using the ALPB implicit solvation model ({SOLVENT}) {cite('alpb')}"
_conf += ". The lowest-energy conformer whose covalent connectivity was preserved was taken forward."
if USE_INPUT_3D:
    _conf += " Where 3D input coordinates were supplied, they were used as the starting geometry."
METHODS.append(_conf)
# built strictly left-to-right so reference numbers follow their order of appearance
_p = f"All density functional theory (DFT) calculations were carried out with PySCF {pyscf.__version__} {cite('pyscf')}"
_p += f" with GPU acceleration through GPU4PySCF {cite('gpu4pyscf', 'gpu4pyscf2')}" if USE_GPU else ""
_p += f" and exchange–correlation functionals from Libxc {cite('libxc')}. "
_p += f"Geometries were optimised at the {_func_txt(CFG['opt_func'], CFG['opt_basis'])}, "
_p += (f"using the geomeTRIC optimiser with translation–rotation internal coordinates {cite('geometric')} and Gaussian-equivalent default "
       f"convergence criteria (maximum force 4.5×10⁻⁴ Eh/bohr, RMS force 3.0×10⁻⁴ Eh/bohr). SCF energies were converged to 10⁻⁹ Eh on a "
       f"level-{4 if 'M06-2X' in (CFG['opt_func'], CFG['sp_func']) else 3} integration grid.")
if USE_DENSITY_FITTING:
    _p += f" The resolution-of-identity (density-fitting) approximation was used for Coulomb and exchange integrals with the def2-universal-JKFIT auxiliary basis {cite('jkfit')}."
if SOLVENT != "none":
    _p += (f" Solvent effects ({SOLVENT}, ε = {SOLVENT_EPS[SOLVENT]}) were included in all DFT calculations through the integral-equation-formalism "
           f"polarizable continuum model (IEF-PCM) {cite('iefpcm', 'pcmrev')}.")
METHODS.append(_p)
if RUN_FREQUENCIES:
    METHODS.append(f"Harmonic vibrational frequencies were obtained from analytic Hessians at the optimisation level to confirm that each structure is a "
                   f"minimum (no imaginary frequencies) and to compute zero-point energies, enthalpies and Gibbs free energies at {TEMPERATURE_K} K and 1 atm "
                   f"within the rigid-rotor/harmonic-oscillator approximation {cite('mcquarrie')}"
                   + (f", with frequencies scaled by {FREQ_SCALE_FACTOR} {cite('cccbdb')}" if FREQ_SCALE_FACTOR != 1.0 else " (unscaled)")
                   + ("." if SOLVENT != "none" else ". Infrared intensities were computed analytically from atomic polar tensors and spectra were broadened with Lorentzian functions (FWHM 20 cm⁻¹)."))
_sp = (f"Electronic properties were computed by single-point calculations at the {_func_txt(CFG['sp_func'], CFG['sp_basis'])} on the optimised geometries. "
       f"Global reactivity descriptors were derived from Koopmans' approximation {cite('koopmans')}" +
       (" and from vertical ΔSCF ionisation energies and electron affinities (N±1 electron systems at fixed geometry)" if RUN_FUKUI else "") +
       f": electronegativity χ = (I+A)/2, chemical hardness η = (I−A)/2 {cite('parrpearson')}, softness S = 1/(2η), electrophilicity ω = χ²/(2η) "
       f"{cite('omega')}, and electron-accepting/donating powers ω⁺ and ω⁻ {cite('gazquez')}.")
METHODS.append(_sp)
if RUN_FUKUI:
    METHODS.append(f"Atomic charges were obtained by Hirshfeld partitioning {cite('hirshfeld')} using spherically averaged free-atom densities computed with the same "
                   f"basis set, together with Mulliken charges {cite('mulliken')}. Condensed Fukui functions {cite('fukui', 'condensed')} f⁺, f⁻ and f⁰ were "
                   f"evaluated by finite differences of Hirshfeld charges of the N, N+1 and N−1 electron systems, and the dual descriptor Δf = f⁺ − f⁻ "
                   f"{cite('dual')} was used to identify electrophilic and nucleophilic sites.")
if RUN_SURFACES:
    METHODS.append(f"The molecular electrostatic potential (MEP) was evaluated exactly on the 0.001 au electron-density isosurface {cite('bader', 'mep')}, "
                   f"extracted from a {CUBE_POINTS}³ density grid by the marching-cubes algorithm {cite('marching', 'skimage')}, and quantified with the "
                   f"Murray–Politzer surface descriptors (V_S,min, V_S,max, Π, σ²_tot, ν) {cite('mpdesc')}. Frontier orbitals were visualised at ±{ISO_ORB} au.")
if RUN_TDDFT:
    METHODS.append(f"Vertical excitation energies and oscillator strengths of the lowest {TD_NSTATES} excited states were calculated by "
                   f"{'TD-DFT within the Tamm–Dancoff approximation ' + cite('tda') if TD_USE_TDA else 'linear-response TD-DFT ' + cite('rungegross', 'casida')} "
                   f"at the {TD_FUNC} {cite(FUNC_REF[TD_FUNC])}/{CFG['sp_basis']} level on the optimised geometries"
                   + (" with IEF-PCM solvation" if SOLVENT != "none" else "") + "; UV-Vis spectra were simulated by Gaussian broadening (σ = 0.4 eV).")
    _cpu = [r["name"] for r in MOLS if RES.get(r["name"], {}).get("tddft", {}).get("engine") == "CPU" and USE_GPU]
    if _cpu:
        METHODS[-1] += f" TD-DFT calculations for {', '.join(_cpu)} were run on CPU."
if RUN_VALIDATION and VALIDATION.get("rows"):
    METHODS.append(f"The protocol was validated against experimental equilibrium structures and dipole moments of water, formaldehyde and ammonia "
                   f"{cite('cccbdb')}; {VALIDATION['n_pass']} of {VALIDATION['n_total']} benchmark quantities were reproduced within tolerance "
                   f"(±0.025 Å, ±2.5°, ±0.25 D).")
METHODS.append(f"The complete workflow was executed with the BinFoLab DFT-Flow notebook (v1.0); 3D structures were inspected with 3Dmol.js {cite('3dmol')}.")
REFERENCE_LIST = [f"{i+1}. {REFS[k]}" for i, k in enumerate(_order)]
METHODS_MD = "\n\n".join(METHODS) + "\n\n### References\n\n" + "\n".join(REFERENCE_LIST)
open(os.path.join(REPORT_DIR, "methods.md"), "w").write(METHODS_MD)
display(Markdown(METHODS_MD))

### Computational methods

Starting structures were generated from SMILES with RDKit [1] using the ETKDGv3 distance-geometry algorithm [2,3] (30 conformers, 0.3 Å RMSD pruning), relaxed with the MMFF94 force field [4], and the 5 lowest conformers were re-optimised at the GFN2-xTB level [5] with tblite through ASE [6] using the ALPB implicit solvation model (DMSO) [7]. The lowest-energy conformer whose covalent connectivity was preserved was taken forward.

All density functional theory (DFT) calculations were carried out with PySCF 2.14.0 [8] with GPU acceleration through GPU4PySCF [9,10] and exchange–correlation functionals from Libxc [11]. Geometries were optimised at the B3LYP-D3(BJ)/def2-SVP level (B3LYP functional [12], Grimme's D3(BJ) dispersion correction [13,14], def2-SVP basis set [15]), using the geomeTRIC optimiser with translation–rotation internal coordinates [16] and Gaussian-equivalent default convergence criteria (maximum force 4.5×10⁻⁴ Eh/bohr, RMS force 3.0×10⁻⁴ Eh/bohr). SCF energies were converged to 10⁻⁹ Eh on a level-3 integration grid. The resolution-of-identity (density-fitting) approximation was used for Coulomb and exchange integrals with the def2-universal-JKFIT auxiliary basis [17]. Solvent effects (DMSO, ε = 46.826) were included in all DFT calculations through the integral-equation-formalism polarizable continuum model (IEF-PCM) [18,19].

Harmonic vibrational frequencies were obtained from analytic Hessians at the optimisation level to confirm that each structure is a minimum (no imaginary frequencies) and to compute zero-point energies, enthalpies and Gibbs free energies at 298.15 K and 1 atm within the rigid-rotor/harmonic-oscillator approximation [20] (unscaled).

Electronic properties were computed by single-point calculations at the B3LYP-D3(BJ)/def2-TZVP level (B3LYP functional [12], Grimme's D3(BJ) dispersion correction [13,14], def2-TZVP basis set [15]) on the optimised geometries. Global reactivity descriptors were derived from Koopmans' approximation [21] and from vertical ΔSCF ionisation energies and electron affinities (N±1 electron systems at fixed geometry): electronegativity χ = (I+A)/2, chemical hardness η = (I−A)/2 [22], softness S = 1/(2η), electrophilicity ω = χ²/(2η) [23], and electron-accepting/donating powers ω⁺ and ω⁻ [24].

Atomic charges were obtained by Hirshfeld partitioning [25] using spherically averaged free-atom densities computed with the same basis set, together with Mulliken charges [26]. Condensed Fukui functions [27,28] f⁺, f⁻ and f⁰ were evaluated by finite differences of Hirshfeld charges of the N, N+1 and N−1 electron systems, and the dual descriptor Δf = f⁺ − f⁻ [29] was used to identify electrophilic and nucleophilic sites.

The molecular electrostatic potential (MEP) was evaluated exactly on the 0.001 au electron-density isosurface [30,31], extracted from a 64³ density grid by the marching-cubes algorithm [32,33], and quantified with the Murray–Politzer surface descriptors (V_S,min, V_S,max, Π, σ²_tot, ν) [34]. Frontier orbitals were visualised at ±0.03 au.

Vertical excitation energies and oscillator strengths of the lowest 10 excited states were calculated by linear-response TD-DFT [35,36] at the CAM-B3LYP [37]/def2-TZVP level on the optimised geometries with IEF-PCM solvation; UV-Vis spectra were simulated by Gaussian broadening (σ = 0.4 eV).

The protocol was validated against experimental equilibrium structures and dipole moments of water, formaldehyde and ammonia [38]; 9 of 10 benchmark quantities were reproduced within tolerance (±0.025 Å, ±2.5°, ±0.25 D).

The complete workflow was executed with the BinFoLab DFT-Flow notebook (v1.0); 3D structures were inspected with 3Dmol.js [39].

### References

1. RDKit: Open-source cheminformatics software. https://www.rdkit.org
2. Riniker, S.; Landrum, G. A. Better informed distance geometry: using what we know to improve conformation generation. J. Chem. Inf. Model. 2015, 55, 2562–2574.
3. Wang, S.; Witek, J.; Landrum, G. A.; Riniker, S. Improving conformer generation for small rings and macrocycles based on distance geometry and experimental torsional-angle preferences. J. Chem. Inf. Model. 2020, 60, 2044–2058.
4. Halgren, T. A. Merck molecular force field. I. Basis, form, scope, parameterization, and performance of MMFF94. J. Comput. Chem. 1996, 17, 490–519.
5. Bannwarth, C.; Ehlert, S.; Grimme, S. GFN2-xTB—An accurate and broadly parametrized self-consistent tight-binding quantum chemical method with multipole electrostatics and density-dependent dispersion contributions. J. Chem. Theory Comput. 2019, 15, 1652–1671.
6. Larsen, A. H. et al. The atomic simulation environment—a Python library for working with atoms. J. Phys.: Condens. Matter 2017, 29, 273002.
7. Ehlert, S.; Stahn, M.; Spicher, S.; Grimme, S. Robust and efficient implicit solvation model for fast semiempirical methods. J. Chem. Theory Comput. 2021, 17, 4250–4261.
8. Sun, Q. et al. Recent developments in the PySCF program package. J. Chem. Phys. 2020, 153, 024109.
9. Li, R.; Sun, Q.; Zhang, X.; Chan, G. K.-L. Introducing GPU acceleration into the Python-based simulations of chemistry framework. J. Phys. Chem. A 2025, 129, 1459–1468.
10. Wu, X.; Sun, Q.; Pu, Z.; Zheng, T.; Ma, W.; Yan, W.; Yu, X.; Wu, Z.; Huo, M.; Li, X.; Ren, W.; Gong, S.; Zhang, Y.; Gao, W. Enhancing GPU-acceleration in the Python-based simulations of chemistry framework. arXiv:2404.09452, 2024.
11. Lehtola, S.; Steigemann, C.; Oliveira, M. J. T.; Marques, M. A. L. Recent developments in libxc—A comprehensive library of functionals for density functional theory. SoftwareX 2018, 7, 1–5.
12. Becke, A. D. Density-functional thermochemistry. III. The role of exact exchange. J. Chem. Phys. 1993, 98, 5648–5652; Lee, C.; Yang, W.; Parr, R. G. Phys. Rev. B 1988, 37, 785–789; Stephens, P. J.; Devlin, F. J.; Chabalowski, C. F.; Frisch, M. J. J. Phys. Chem. 1994, 98, 11623–11627.
13. Grimme, S.; Antony, J.; Ehrlich, S.; Krieg, H. A consistent and accurate ab initio parametrization of density functional dispersion correction (DFT-D) for the 94 elements H–Pu. J. Chem. Phys. 2010, 132, 154104.
14. Grimme, S.; Ehrlich, S.; Goerigk, L. Effect of the damping function in dispersion corrected density functional theory. J. Comput. Chem. 2011, 32, 1456–1465.
15. Weigend, F.; Ahlrichs, R. Balanced basis sets of split valence, triple zeta valence and quadruple zeta valence quality for H to Rn: design and assessment of accuracy. Phys. Chem. Chem. Phys. 2005, 7, 3297–3305.
16. Wang, L.-P.; Song, C. Geometry optimization made simple with translation and rotation coordinates. J. Chem. Phys. 2016, 144, 214108.
17. Weigend, F. Accurate Coulomb-fitting basis sets for H to Rn. Phys. Chem. Chem. Phys. 2006, 8, 1057–1065.
18. Cancès, E.; Mennucci, B.; Tomasi, J. A new integral equation formalism for the polarizable continuum model: theoretical background and applications to isotropic and anisotropic dielectrics. J. Chem. Phys. 1997, 107, 3032–3041.
19. Tomasi, J.; Mennucci, B.; Cammi, R. Quantum mechanical continuum solvation models. Chem. Rev. 2005, 105, 2999–3094.
20. McQuarrie, D. A. Statistical Mechanics; University Science Books: Sausalito, CA, 2000.
21. Koopmans, T. Über die Zuordnung von Wellenfunktionen und Eigenwerten zu den einzelnen Elektronen eines Atoms. Physica 1934, 1, 104–113.
22. Parr, R. G.; Pearson, R. G. Absolute hardness: companion parameter to absolute electronegativity. J. Am. Chem. Soc. 1983, 105, 7512–7516.
23. Parr, R. G.; Szentpály, L. v.; Liu, S. Electrophilicity index. J. Am. Chem. Soc. 1999, 121, 1922–1924.
24. Gázquez, J. L.; Cedillo, A.; Vela, A. Electrodonating and electroaccepting powers. J. Phys. Chem. A 2007, 111, 1966–1970.
25. Hirshfeld, F. L. Bonded-atom fragments for describing molecular charge densities. Theor. Chim. Acta 1977, 44, 129–138.
26. Mulliken, R. S. Electronic population analysis on LCAO–MO molecular wave functions. I. J. Chem. Phys. 1955, 23, 1833–1840.
27. Parr, R. G.; Yang, W. Density functional approach to the frontier-electron theory of chemical reactivity. J. Am. Chem. Soc. 1984, 106, 4049–4050.
28. Yang, W.; Mortier, W. J. The use of global and local molecular parameters for the analysis of the gas-phase basicity of amines. J. Am. Chem. Soc. 1986, 108, 5708–5711.
29. Morell, C.; Grand, A.; Toro-Labbé, A. New dual descriptor for chemical reactivity. J. Phys. Chem. A 2005, 109, 205–212.
30. Bader, R. F. W.; Carroll, M. T.; Cheeseman, J. R.; Chang, C. Properties of atoms in molecules: atomic volumes. J. Am. Chem. Soc. 1987, 109, 7968–7979.
31. Murray, J. S.; Politzer, P. The electrostatic potential: an overview. WIREs Comput. Mol. Sci. 2011, 1, 153–163.
32. Lewiner, T.; Lopes, H.; Vieira, A. W.; Tavares, G. Efficient implementation of Marching Cubes' cases with topological guarantees. J. Graphics Tools 2003, 8, 1–15.
33. van der Walt, S. et al. scikit-image: image processing in Python. PeerJ 2014, 2, e453.
34. Murray, J. S.; Lane, P.; Brinck, T.; Paulsen, K.; Grice, M. E.; Politzer, P. Relationships of critical constants and boiling points to computed molecular surface properties. J. Phys. Chem. 1993, 97, 9369–9373.
35. Runge, E.; Gross, E. K. U. Density-functional theory for time-dependent systems. Phys. Rev. Lett. 1984, 52, 997–1000.
36. Casida, M. E. Time-dependent density functional response theory for molecules. In Recent Advances in Density Functional Methods, Part I; Chong, D. P., Ed.; World Scientific: Singapore, 1995; pp 155–192.
37. Yanai, T.; Tew, D. P.; Handy, N. C. A new hybrid exchange–correlation functional using the Coulomb-attenuating method (CAM-B3LYP). Chem. Phys. Lett. 2004, 393, 51–57.
38. Johnson, R. D., III, Ed. NIST Computational Chemistry Comparison and Benchmark Database, NIST Standard Reference Database 101. https://cccbdb.nist.gov/
39. Rego, N.; Koes, D. 3Dmol.js: molecular visualization with WebGL. Bioinformatics 2015, 31, 1322–1324.

### 📘 For Your Read — Cell 14.3: PDF report

**What it does.** Builds `report/DFTFlow_report.pdf`: run information and level of theory, validation outcome, the master summary, every figure with its caption, the methods text and references. Figures are embedded at reduced resolution to keep the PDF small; the 600 DPI originals stay in `figures/`.

In [ ]:
#@title Cell 14.3 — PDF report
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image as RLImage, PageBreak
from PIL import Image as PILImage
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.lib.fonts import addMapping
# Unicode font (Greek letters, super/subscripts) shipped with Matplotlib
_fd = os.path.join(matplotlib.get_data_path(), "fonts", "ttf")
for _nm, _f in (("DejaVu", "DejaVuSans.ttf"), ("DejaVu-Bold", "DejaVuSans-Bold.ttf"),
                ("DejaVu-Oblique", "DejaVuSans-Oblique.ttf"), ("DejaVu-BoldOblique", "DejaVuSans-BoldOblique.ttf")):
    pdfmetrics.registerFont(TTFont(_nm, os.path.join(_fd, _f)))
addMapping("DejaVu", 0, 0, "DejaVu"); addMapping("DejaVu", 1, 0, "DejaVu-Bold")
addMapping("DejaVu", 0, 1, "DejaVu-Oblique"); addMapping("DejaVu", 1, 1, "DejaVu-BoldOblique")
FONT = "DejaVu"

def _esc(s):
    return str(s).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")

def _md_inline(s):
    s = _esc(s)
    s = re.sub(r"\*\*(.+?)\*\*", r"<b>\1</b>", s)
    s = re.sub(r"`(.+?)`", r"<font face='Courier'>\1</font>", s)
    return s

def _rl_table(df_, cols, widths=None, fs=6.5):
    data = [[Paragraph(f"<b>{_esc(c)}</b>", ParagraphStyle("h", fontName=FONT, fontSize=fs, leading=fs + 1.5)) for c in cols]]
    for _, r in df_[cols].iterrows():
        data.append([Paragraph(_esc(f"{v:.4g}" if isinstance(v, float) else v), ParagraphStyle("c", fontName=FONT, fontSize=fs, leading=fs + 1.5)) for v in r])
    t = Table(data, colWidths=widths, repeatRows=1)
    t.setStyle(TableStyle([("GRID", (0, 0), (-1, -1), 0.25, colors.HexColor("#c9c8c3")),
                           ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#eef3fb")), ("VALIGN", (0, 0), (-1, -1), "TOP")]))
    return t

ss = getSampleStyleSheet()
H1 = ParagraphStyle("H1", parent=ss["Heading1"], fontName="DejaVu-Bold", fontSize=16, textColor=colors.HexColor("#1c5cab"))
H2 = ParagraphStyle("H2", parent=ss["Heading2"], fontName="DejaVu-Bold", fontSize=11.5, textColor=colors.HexColor("#1c5cab"))
BODY = ParagraphStyle("B", parent=ss["BodyText"], fontName=FONT, fontSize=8.5, leading=11.5)
CAP = ParagraphStyle("C", parent=ss["BodyText"], fontName=FONT, fontSize=7.5, leading=9.5, textColor=colors.HexColor("#52514e"))
pdf_path = os.path.join(REPORT_DIR, "DFTFlow_report.pdf")
doc = SimpleDocTemplate(pdf_path, pagesize=A4, leftMargin=1.6 * cm, rightMargin=1.6 * cm, topMargin=1.5 * cm, bottomMargin=1.5 * cm)
W = A4[0] - 3.2 * cm
story = [Paragraph("DFT-Flow Report — BinFoLab", H1),
         Paragraph(f"Project <b>{_esc(PROJECT_NAME)}</b> · generated {datetime.datetime.now():%Y-%m-%d %H:%M}", BODY), Spacer(1, 6),
         Paragraph(f"<b>Level of theory:</b> {_esc(LEVEL_STRING)}", BODY),
         Paragraph(f"<b>TD-DFT:</b> {_esc(TD_FUNC)}/{_esc(CFG['sp_basis'])}, {TD_NSTATES} states · <b>Backend:</b> {_esc(CFG['backend'])}", BODY),
         Paragraph(f"<b>Molecules:</b> {len(MOLS)} · <b>Temperature:</b> {TEMPERATURE_K} K · <b>Frequency scale:</b> {FREQ_SCALE_FACTOR}", BODY)]
if VALIDATION.get("rows"):
    story.append(Paragraph(f"<b>Validation:</b> {VALIDATION['n_pass']}/{VALIDATION['n_total']} benchmark quantities within tolerance.", BODY))
_fail = {r["name"]: list(RES.get(r["name"], {}).get("errors", {}).keys()) for r in MOLS if RES.get(r["name"], {}).get("errors")}
if _fail:
    story.append(Paragraph("<b>Stages with errors:</b> " + _esc("; ".join(f"{k}: {', '.join(v)}" for k, v in _fail.items())), BODY))
if HIGHLIGHTS:
    story += [Spacer(1, 6), Paragraph("Computed highlights", H2)] + [Paragraph(_md_inline(h.lstrip("- ")), BODY) for h in HIGHLIGHTS]
story += [Spacer(1, 6), Paragraph("Summary", H2)]
_cols = [c for c in ["molecule", "formula", "HOMO (eV)", "LUMO (eV)", "gap (eV)", "η (eV)", "ω (eV)", "dipole (D)",
                     "V_S,min (kcal/mol)", "λmax (nm)", "n imag"] if c in MASTER]
story.append(_rl_table(MASTER, _cols))
story += [PageBreak(), Paragraph("Figures", H2)]
_tmpd = os.path.join(REPORT_DIR, "_img"); os.makedirs(_tmpd, exist_ok=True)
for k, f in enumerate(sorted(FIGURES, key=lambda x: (x["part"], x["file"]))):
    if not os.path.exists(f["file"]):
        continue
    im = PILImage.open(f["file"]).convert("RGB")
    if im.width > 1800:
        im = im.resize((1800, int(im.height * 1800 / im.width)), PILImage.LANCZOS)
    p = os.path.join(_tmpd, f"fig{k}.jpg"); im.save(p, quality=88)
    h = W * im.height / im.width
    if h > 22 * cm:
        h = 22 * cm
    story += [RLImage(p, width=h * im.width / im.height, height=h), Paragraph(f"<b>Figure {k+1}.</b> {_esc(f['caption'])}", CAP), Spacer(1, 10)]
story.append(PageBreak())
for block in METHODS_MD.split("\n\n"):
    if block.startswith("### "):
        story.append(Paragraph(_esc(block[4:]), H2))
    elif block.strip():
        for ln in block.split("\n"):
            story.append(Paragraph(_md_inline(ln), CAP if re.match(r"^\d+\. ", ln) else BODY))
doc.build(story)
shutil.rmtree(_tmpd, ignore_errors=True)
print("PDF report:", pdf_path, f"({os.path.getsize(pdf_path)/1e6:.1f} MB)")

PDF report: /content/DFTFlow_run1/report/DFTFlow_report.pdf (1.7 MB)


### 📘 For Your Read — Cell 14.4: Package and download

**What it does.** Zips the project folder (figures, tables, structures, Molden files, report, configuration, state; cube files optional) and downloads it in Colab. Large checkpoint arrays are left out by default; they live in the project folder (and on Drive if enabled) for resuming.

In [ ]:
#@title Cell 14.4 — ZIP & download
INCLUDE_CUBES = True  #@param {type:"boolean"}
INCLUDE_CHECKPOINTS = False  #@param {type:"boolean"}
_zip = os.path.join(os.path.dirname(OUT_DIR), f"{safe_name(PROJECT_NAME)}_DFTFlow_{datetime.datetime.now():%Y%m%d_%H%M}.zip")
n_files = 0
with zipfile.ZipFile(_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, fs in os.walk(OUT_DIR):
        rel = os.path.relpath(root, OUT_DIR)
        if (rel.startswith("checkpoints") and not INCLUDE_CHECKPOINTS) or (rel.startswith("cubes") and not INCLUDE_CUBES):
            continue
        for f in fs:
            z.write(os.path.join(root, f), os.path.join(safe_name(PROJECT_NAME), rel, f)); n_files += 1
print(f"📦 {n_files} files → {_zip} ({os.path.getsize(_zip)/1e6:.1f} MB)")
_err = {r["name"]: list(RES.get(r["name"], {}).get("errors", {}).keys()) for r in MOLS if RES.get(r["name"], {}).get("errors")}
if _err:
    print("⚠️  Stages that failed (see state.json → errors for tracebacks):", _err)
if IN_COLAB:
    colab_files.download(_zip)

📦 57 files → /content/DFTFlow_run1_DFTFlow_20261025_1704.zip (21.2 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## Part 15 — Notes, Troubleshooting and Citation

#### Troubleshooting
| Symptom | Likely cause | What to do |
|---|---|---|
| *SCF did not converge* | Difficult electronic structure (zwitterion, metal, near-degenerate radical) | Check charge/multiplicity; try PBE0 or Fast preset; for radicals confirm the multiplicity. |
| Optimisation not converged | Very flat torsions | Re-run Cell 7.1 (it continues from the checkpoint); results are usable if the frequency check passes. |
| Imaginary frequency remains | Saddle point or flat surface | Inspect the mode in a viewer; re-run with `RERUN_STAGES = "opt,freq"` from a different conformer (raise `N_CONFORMERS`). |
| Colab disconnected | Session limits | Re-run from the top. Finished stages are skipped, interrupted optimisations resume. Enable *Save to Google Drive* for long runs. |
| GPU out of memory | Molecule too large for the card | Use def2-SVP for the single point, keep density fitting on, or move to an L4/A100 runtime. |
| Wrong protonation state | SMILES as written | DFT keeps your protonation; supply the intended microspecies (e.g. from a pKa tool). |

#### Interpretation caveats
- All properties are for **one conformer** at **0 K electronic / harmonic** level with **implicit** solvent. Hydrogen-bonded or ion-paired environments need explicit-solvent or QM/MM treatments.
- Koopmans-based descriptors depend noticeably on the functional (exact-exchange fraction); compare molecules only at the same level of theory.
- Condensed Fukui values depend on the charge partitioning scheme; Hirshfeld is used because it is the most transferable, but report it.

#### How to cite
Cite the method papers listed in the auto-generated references (Cell 14.2), which include PySCF, GPU4PySCF, geomeTRIC, GFN2-xTB and the functionals and basis sets used, and the BinFoLab DFT-Flow notebook.

